<a href="https://colab.research.google.com/github/ornab74/zom-nom-defense/blob/main/Universal_PolyFlow_Asset_Compiler.For.Google.Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal PolyFlow Asset Compiler — GPT‑5.6 + GPT Image + Blender + Godot

This Colab is a secure, prompt-driven game-asset compiler.

- **Default preset:** the advanced PolyFlow/RigFlow procedural tree.
- **Universal mode:** change `ASSET_PROMPT` to a vehicle, building, prop,
  creature, foliage set, furniture item, environmental kit, or other game asset.
- **GPT‑5.6 planner:** produces a strict `AssetGraph` JSON plan, never executable code.
- **GPT Image texture foundry:** creates coherent material families and multiple
  physically scaled leaf textures.
- **Trusted compiler:** validated AssetGraph nodes are compiled to Blender by
  repository-owned code.
- **Exports:** GLB, Blender source, render, previews, manifests, checksums, and ZIP.
- **No Pillow:** image work uses NumPy + OpenCV.


## 0. Runtime setup

In Colab, choose **Runtime → Change runtime type → T4 GPU**. The procedural mesh and topology stages run on CPU; Blender rendering can use GPU when available. The GPT Image step requires an OpenAI API key, but a procedural fallback is included.


## Colab runtime recovery

This version never upgrades Colab's numerical stack during normal setup.

If an older notebook already upgraded NumPy in the current session, run
the setup cell below. It will repair the incompatible packages and restart
the runtime once. After Colab reconnects, choose **Runtime → Run all**.

If the runtime was heavily modified by another notebook, use
**Runtime → Disconnect and delete runtime**, reconnect, and run this
notebook from the beginning.


In [ ]:
#@title Safe Colab setup: preserve NumPy/SciPy ABI compatibility
# This cell deliberately DOES NOT upgrade NumPy, SciPy, Matplotlib, or OpenCV.
# Replacing those packages inside a live Colab kernel can mix newly installed
# Python files with already-loaded compiled extensions and cause errors such as:
#   AttributeError: ... _multiarray_umath has no attribute _blas_supports_fpe

!apt-get -qq update
!apt-get -qq install -y blender ffmpeg libegl1 > /dev/null

# Install only the packages that the notebook adds. "only-if-needed" prevents
# pip from replacing Colab's working numerical stack when its versions already
# satisfy dependencies.
%pip install -q --upgrade-strategy only-if-needed openai pygltflib trimesh networkx

import os
import subprocess
import sys
from pathlib import Path

NUMERIC_REPAIR_MARKER = Path("/content/.polyflow_numeric_stack_repaired")

probe_code = r"""
import numpy as np
import numpy.testing
import scipy
import scipy.sparse
import matplotlib
import cv2
print("NumPy", np.__version__)
print("SciPy", scipy.__version__)
print("Matplotlib", matplotlib.__version__)
print("OpenCV", cv2.__version__)
"""

probe = subprocess.run(
    [sys.executable, "-c", probe_code],
    text=True,
    capture_output=True,
)

if probe.returncode != 0:
    print("The Colab numerical stack is inconsistent.")
    print(probe.stderr[-4000:])

    if NUMERIC_REPAIR_MARKER.exists():
        raise RuntimeError(
            "The numerical stack is still inconsistent after repair. "
            "Use Runtime > Disconnect and delete runtime, reconnect, then run "
            "this notebook from the first cell."
        )

    # This compatibility set supports Python 3.12 and is installed only when
    # the existing environment has already become corrupted.
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--force-reinstall",
            "--no-cache-dir",
            "numpy==1.26.4",
            "scipy==1.13.1",
            "matplotlib==3.8.4",
            "opencv-python-headless==4.10.0.84",
        ]
    )
    NUMERIC_REPAIR_MARKER.write_text("repaired\n", encoding="utf-8")
    print(
        "\nA consistent NumPy/SciPy/OpenCV stack was installed. "
        "Colab will restart once now. After reconnecting, run all cells again."
    )
    os.kill(os.getpid(), 9)
else:
    print(probe.stdout)
    print("Setup complete. Numerical ABI probe passed.")


In [ ]:
#@title Imports, directories, and reproducibility
from __future__ import annotations

import os, json, math, base64, textwrap, subprocess, shutil, zipfile, hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
from getpass import getpass
from IPython.display import display, Image as IPyImage

import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import sparse
from scipy.spatial import cKDTree
from scipy.sparse.linalg import eigsh

ROOT = Path('/content/tree_asset')
ROOT.mkdir(parents=True, exist_ok=True)

# Exercise NumPy's testing extension immediately so ABI problems fail here
# with a clear setup-cell diagnosis rather than much later in mesh generation.
import numpy.testing as _numpy_testing

np.set_printoptions(precision=4, suppress=True)
print('Output directory:', ROOT)
print('Image backend: NumPy + OpenCV (Pillow removed)')
print('Numerical stack:', {
    'numpy': np.__version__,
    'opencv': cv2.__version__,
})


## 1. Asset configuration

The geometry is deterministic under `seed`. Higher values of `attractor_count`, `growth_iterations`, and `radial_segments` improve density but increase topology-analysis and rendering cost.


In [ ]:
#@title Universal asset request, security policy, and default tree setup
from dataclasses import dataclass, asdict, field
from typing import Any
import re

# Change only this prompt to redesign the output. The tree remains the default.
ASSET_PROMPT = """
A beautiful mature southeastern live oak for a premium Godot survival game.
Broad asymmetrical crown, believable branch hierarchy, old fissured bark,
multiple leaf sizes, animation-ready wind rig, four LOD targets, clean collision,
grounded realistic proportions, optimized for a three-quarter tactical camera.
""".strip()

# GPT planner and image model IDs are configurable because API access differs
# between accounts. The notebook checks the live model list before requesting.
PLANNER_MODEL = "gpt-5.6"
PLANNER_FALLBACK_MODELS = ["gpt-5.2", "gpt-5"]
ALLOW_PLANNER_FALLBACK = True
IMAGE_MODEL = "gpt-image-2"

USE_GPT_PLANNER = True
USE_GPT_IMAGES = True

@dataclass
class SecurityPolicy:
    max_prompt_chars: int = 8000
    max_parts: int = 384
    max_materials: int = 32
    max_texture_variants_per_material: int = 8
    max_dimension_m: float = 120.0
    max_total_estimated_triangles: int = 1_500_000
    max_api_image_bytes: int = 64 * 1024 * 1024
    allow_model_fallback: bool = True
    execute_model_python: bool = False
    allow_remote_urls: bool = False
    allowed_primitives: tuple[str, ...] = (
        "cube", "cylinder", "sphere", "uv_sphere", "ico_sphere",
        "cone", "plane", "torus", "capsule",
    )
    allowed_modifiers: tuple[str, ...] = (
        "bevel", "subdivision", "mirror", "solidify", "array",
        "weighted_normal", "decimate",
    )
    allowed_categories: tuple[str, ...] = (
        "tree", "foliage", "prop", "vehicle", "architecture",
        "character", "creature", "furniture", "environment",
        "defense", "modular_kit",
    )

@dataclass
class TreeConfig:
    seed: int = 42
    height: float = 8.0
    trunk_nodes: int = 22
    branch_levels: int = 11
    branches_per_level: int = 2
    attractor_count: int = 1000
    growth_iterations: int = 52
    influence_radius: float = 1.60
    kill_radius: float = 0.32
    step_size: float = 0.24
    radial_segments: int = 10
    base_radius: float = 0.48
    tip_radius: float = 0.014
    crown_radius: float = 3.40
    crown_height: float = 4.80
    max_bones: int = 240
    max_leaves: int = 520
    leaf_texture_variants: int = 6

@dataclass
class RenderConfig:
    resolution_x: int = 1280
    resolution_y: int = 720
    samples: int = 64
    transparent_background: bool = False
    engine: str = "EEVEE"
    create_turntable: bool = False
    turntable_frames: int = 120

SECURITY = SecurityPolicy()
CFG = TreeConfig()
RENDER = RenderConfig()

if not ASSET_PROMPT or len(ASSET_PROMPT) > SECURITY.max_prompt_chars:
    raise ValueError(
        f"ASSET_PROMPT must contain 1–{SECURITY.max_prompt_chars} characters"
    )

(ROOT / "request.json").write_text(
    json.dumps(
        {
            "asset_prompt": ASSET_PROMPT,
            "planner_model": PLANNER_MODEL,
            "image_model": IMAGE_MODEL,
            "security": asdict(SECURITY),
            "tree": asdict(CFG),
            "render": asdict(RENDER),
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Asset request:")
print(ASSET_PROMPT)
print("\nSecurity: model-written Python execution =", SECURITY.execute_model_python)


## Secure GPT‑5.6 AssetGraph planner

The model plans an asset using strict JSON. It cannot inject Python,
shell commands, file paths, URLs, Blender operators, or arbitrary code.

The trusted compiler minimizes the following conceptual objective:

\[
\mathcal{L}_{asset} =
\lambda_{fm}\mathbb{E}\lVert v_\theta(x_t,t,c)-(x_1-x_0)\rVert_2^2 +
\lambda_{ch}d_{Chamfer} +
\lambda_{lap}\lVert LV\rVert_2^2 +
\lambda_e\sum_{(i,j)\in E}(\lVert v_i-v_j\rVert-\ell_{ij}^{*})^2 +
\lambda_n\mathcal{L}_{normal} +
\lambda_m\mathcal{L}_{manifold} +
\lambda_c\mathcal{L}_{collision}.
\]

For rigged assets, skin weights use bounded normalized influences:

\[
w_{ib} =
\frac{\exp(-d_g(v_i,b)^2/\tau)}
{\sum_{k\in\mathcal{N}_B(i)}\exp(-d_g(v_i,k)^2/\tau)},
\qquad \sum_b w_{ib}=1.
\]


In [ ]:
#@title Plan any asset with GPT‑5.6 Structured Outputs or a secure local fallback
import math
import hashlib

ASSET_GRAPH_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "asset_id", "display_name", "category", "strategy", "dimensions_m",
        "art_direction", "materials", "parts", "rig", "animations",
        "collision", "lod_ratios", "texture_policy",
    ],
    "properties": {
        "asset_id": {"type": "string", "pattern": "^[a-z0-9_]{1,64}$"},
        "display_name": {"type": "string", "maxLength": 120},
        "category": {"type": "string"},
        "strategy": {"type": "string", "enum": ["tree_polyflow", "asset_graph"]},
        "dimensions_m": {
            "type": "array", "minItems": 3, "maxItems": 3,
            "items": {"type": "number"},
        },
        "art_direction": {"type": "string", "maxLength": 2000},
        "materials": {
            "type": "array", "maxItems": 32,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "id", "description", "base_color", "metallic",
                    "roughness", "transparent", "texture_variants",
                ],
                "properties": {
                    "id": {"type": "string", "pattern": "^[a-z0-9_]{1,48}$"},
                    "description": {"type": "string", "maxLength": 1200},
                    "base_color": {
                        "type": "array", "minItems": 3, "maxItems": 3,
                        "items": {"type": "number"},
                    },
                    "metallic": {"type": "number"},
                    "roughness": {"type": "number"},
                    "transparent": {"type": "boolean"},
                    "texture_variants": {"type": "integer"},
                },
            },
        },
        "parts": {
            "type": "array", "maxItems": 384,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "id", "primitive", "location", "rotation_deg", "scale",
                    "material_id", "parent_id", "modifiers",
                ],
                "properties": {
                    "id": {"type": "string", "pattern": "^[a-z0-9_]{1,64}$"},
                    "primitive": {"type": "string"},
                    "location": {
                        "type": "array", "minItems": 3, "maxItems": 3,
                        "items": {"type": "number"},
                    },
                    "rotation_deg": {
                        "type": "array", "minItems": 3, "maxItems": 3,
                        "items": {"type": "number"},
                    },
                    "scale": {
                        "type": "array", "minItems": 3, "maxItems": 3,
                        "items": {"type": "number"},
                    },
                    "material_id": {"type": "string"},
                    "parent_id": {"type": ["string", "null"]},
                    "modifiers": {
                        "type": "array", "maxItems": 8,
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": ["type", "strength"],
                            "properties": {
                                "type": {"type": "string"},
                                "strength": {"type": "number"},
                            },
                        },
                    },
                },
            },
        },
        "rig": {
            "type": "object",
            "additionalProperties": False,
            "required": ["type", "bone_count", "notes"],
            "properties": {
                "type": {"type": "string", "enum": ["none", "wind", "hinge", "humanoid", "custom"]},
                "bone_count": {"type": "integer"},
                "notes": {"type": "string", "maxLength": 1000},
            },
        },
        "animations": {
            "type": "array", "maxItems": 16,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["name", "duration_s", "loop", "description"],
                "properties": {
                    "name": {"type": "string", "maxLength": 64},
                    "duration_s": {"type": "number"},
                    "loop": {"type": "boolean"},
                    "description": {"type": "string", "maxLength": 500},
                },
            },
        },
        "collision": {"type": "string", "enum": ["none", "box", "convex", "trimesh_static", "capsule"]},
        "lod_ratios": {
            "type": "array", "minItems": 1, "maxItems": 5,
            "items": {"type": "number"},
        },
        "texture_policy": {
            "type": "object",
            "additionalProperties": False,
            "required": ["resolution", "pbr_channels", "style_consistency", "leaf_profiles"],
            "properties": {
                "resolution": {"type": "integer"},
                "pbr_channels": {
                    "type": "array", "maxItems": 8,
                    "items": {"type": "string"},
                },
                "style_consistency": {"type": "string", "maxLength": 1200},
                "leaf_profiles": {
                    "type": "array", "maxItems": 8,
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["id", "physical_length_m", "usage", "prompt"],
                        "properties": {
                            "id": {"type": "string"},
                            "physical_length_m": {"type": "number"},
                            "usage": {"type": "string"},
                            "prompt": {"type": "string"},
                        },
                    },
                },
            },
        },
    },
}

PLANNER_SYSTEM_PROMPT = r"""
You are the planning stage of a secure universal game-asset compiler.
Return only an AssetGraph matching the supplied JSON Schema.

Do not emit Python, shell, URLs, imports, code snippets, Blender operators,
filesystem paths, secrets, or instructions to download third-party assets.

Choose strategy tree_polyflow only when the requested subject is fundamentally
a tree or woody branching plant. Otherwise choose asset_graph and represent the
asset as a coherent hierarchy of approved primitives.

Design for Godot 4 GLB import, meter scale, Y-up export, clean origins, readable
top-down silhouette, physically plausible materials, bounded triangle count,
separate moving parts, collision, LODs, and animation hooks.

Think using these geometric constraints:
x_t=(1-t)x_0+t x_1
L_FM=E||v_theta(x_t,t,c)-(x_1-x_0)||^2
L_total=lambda_fm L_FM + lambda_ch L_Chamfer + lambda_lap ||LV||^2
        + lambda_edge L_edge + lambda_normal L_normal
        + lambda_manifold L_manifold + lambda_collision L_collision.

For foliage, provide leaf profiles at distinct physical sizes and purposes.
For hard-surface assets, prioritize separate panels, believable dimensions,
bevel-ready edges, and logical pivots.
For architecture, use modular parts and real wall/floor thickness.
For characters or creatures, provide a safe primitive proxy plan and rig notes;
do not pretend the primitive proxy is a final production character.
"""


def _slug(text: str, fallback: str = "generated_asset") -> str:
    value = re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")
    return (value[:64] or fallback)


def _tree_leaf_profiles() -> list[dict[str, Any]]:
    base_length = max(0.035, min(0.28, CFG.crown_radius / max(math.sqrt(CFG.max_leaves), 1.0)))
    profiles = [
        ("micro", 0.58, "distant crown fill and small terminal shoots"),
        ("small", 0.76, "dense secondary twigs"),
        ("medium", 1.00, "primary gameplay foliage"),
        ("large", 1.28, "foreground silhouette accents"),
        ("damaged", 1.00, "weathered and insect-damaged variation"),
        ("cluster", 1.65, "optimized multi-leaf cluster card for distant LODs"),
    ]
    result = []
    for profile_id, multiplier, usage in profiles[:CFG.leaf_texture_variants]:
        physical = round(base_length * multiplier, 4)
        result.append({
            "id": profile_id,
            "physical_length_m": physical,
            "usage": usage,
            "prompt": (
                f"One isolated mature live-oak foliage reference for a game asset; "
                f"target physical blade or cluster length {physical:.3f} meters; "
                f"{usage}; top-down botanical material capture; transparent background; "
                "complete silhouette; realistic branching veins; natural asymmetry; "
                "no shadows, no text, no extra disconnected objects."
            ),
        })
    return result


def local_asset_plan(prompt: str) -> dict[str, Any]:
    lowered = prompt.lower()
    is_tree = any(word in lowered for word in ("tree", "oak", "pine", "maple", "willow", "palm"))
    asset_id = _slug(prompt.splitlines()[0][:72])
    if is_tree:
        return {
            "asset_id": asset_id,
            "display_name": "PolyFlow Procedural Tree",
            "category": "tree",
            "strategy": "tree_polyflow",
            "dimensions_m": [CFG.crown_radius * 2, CFG.crown_radius * 2, CFG.height],
            "art_direction": prompt,
            "materials": [
                {
                    "id": "bark",
                    "description": "Mature fissured gray-brown bark with restrained moss traces",
                    "base_color": [0.24, 0.14, 0.08],
                    "metallic": 0.0,
                    "roughness": 0.82,
                    "transparent": False,
                    "texture_variants": 2,
                },
                {
                    "id": "leaf",
                    "description": "Deep green live-oak foliage with natural value variation",
                    "base_color": [0.12, 0.35, 0.10],
                    "metallic": 0.0,
                    "roughness": 0.62,
                    "transparent": True,
                    "texture_variants": CFG.leaf_texture_variants,
                },
            ],
            "parts": [],
            "rig": {"type": "wind", "bone_count": CFG.max_bones, "notes": "PolyFlow geodesic wind rig"},
            "animations": [
                {"name": "wind_idle", "duration_s": 4.0, "loop": True, "description": "Layered low-amplitude branch sway"}
            ],
            "collision": "trimesh_static",
            "lod_ratios": [1.0, 0.5, 0.22, 0.08],
            "texture_policy": {
                "resolution": 2048,
                "pbr_channels": ["albedo", "normal", "roughness", "ao", "opacity"],
                "style_consistency": "Grounded stylized realism with neutral material lighting",
                "leaf_profiles": _tree_leaf_profiles(),
            },
        }

    # Secure generic fallback: a coherent proxy assembled from bounded parts.
    category = "vehicle" if any(w in lowered for w in ("car", "truck", "vehicle", "motorcycle")) else "prop"
    return {
        "asset_id": asset_id,
        "display_name": "Universal Prompt Asset",
        "category": category,
        "strategy": "asset_graph",
        "dimensions_m": [3.0, 2.0, 1.8],
        "art_direction": prompt,
        "materials": [
            {
                "id": "primary",
                "description": "Primary weathered surface matching the requested object",
                "base_color": [0.18, 0.42, 0.40],
                "metallic": 0.15,
                "roughness": 0.55,
                "transparent": False,
                "texture_variants": 2,
            },
            {
                "id": "secondary",
                "description": "Dark structural and trim material",
                "base_color": [0.06, 0.07, 0.08],
                "metallic": 0.35,
                "roughness": 0.45,
                "transparent": False,
                "texture_variants": 1,
            },
        ],
        "parts": [
            {
                "id": "body", "primitive": "cube",
                "location": [0, 0, 0.8], "rotation_deg": [0, 0, 0],
                "scale": [1.5, 0.85, 0.55], "material_id": "primary",
                "parent_id": None,
                "modifiers": [{"type": "bevel", "strength": 0.08}],
            },
            {
                "id": "upper", "primitive": "cube",
                "location": [0.1, 0, 1.45], "rotation_deg": [0, 0, 0],
                "scale": [0.85, 0.72, 0.42], "material_id": "primary",
                "parent_id": "body",
                "modifiers": [{"type": "bevel", "strength": 0.06}],
            },
        ],
        "rig": {"type": "none", "bone_count": 0, "notes": "Static proxy"},
        "animations": [],
        "collision": "convex",
        "lod_ratios": [1.0, 0.5, 0.22],
        "texture_policy": {
            "resolution": 2048,
            "pbr_channels": ["albedo", "normal", "roughness", "ao", "metallic"],
            "style_consistency": "Grounded stylized realism with readable game silhouette",
            "leaf_profiles": [],
        },
    }


def _finite_triplet(value: Any) -> bool:
    return (
        isinstance(value, list) and len(value) == 3
        and all(isinstance(v, (int, float)) and math.isfinite(float(v)) for v in value)
    )


def validate_asset_graph(plan: dict[str, Any]) -> dict[str, Any]:
    required = set(ASSET_GRAPH_SCHEMA["required"])
    missing = required - set(plan)
    if missing:
        raise ValueError(f"AssetGraph missing fields: {sorted(missing)}")

    if not re.fullmatch(r"[a-z0-9_]{1,64}", str(plan["asset_id"])):
        raise ValueError("Unsafe asset_id")
    if plan["category"] not in SECURITY.allowed_categories:
        raise ValueError(f"Unsupported category: {plan['category']}")
    if plan["strategy"] not in ("tree_polyflow", "asset_graph"):
        raise ValueError("Unsupported strategy")
    if not _finite_triplet(plan["dimensions_m"]):
        raise ValueError("dimensions_m must be three finite numbers")
    if any(abs(float(v)) > SECURITY.max_dimension_m for v in plan["dimensions_m"]):
        raise ValueError("Asset dimensions exceed security policy")

    materials = plan.get("materials", [])
    parts = plan.get("parts", [])
    if not 1 <= len(materials) <= SECURITY.max_materials:
        raise ValueError("Invalid material count")
    if len(parts) > SECURITY.max_parts:
        raise ValueError("Part count exceeds security policy")

    material_ids = set()
    for material in materials:
        material_id = str(material["id"])
        if not re.fullmatch(r"[a-z0-9_]{1,48}", material_id):
            raise ValueError(f"Unsafe material id: {material_id}")
        if material_id in material_ids:
            raise ValueError(f"Duplicate material id: {material_id}")
        material_ids.add(material_id)
        material["metallic"] = float(np.clip(material["metallic"], 0, 1))
        material["roughness"] = float(np.clip(material["roughness"], 0, 1))
        material["texture_variants"] = int(np.clip(
            material["texture_variants"], 1, SECURITY.max_texture_variants_per_material
        ))
        if not _finite_triplet(material["base_color"]):
            raise ValueError(f"Invalid base_color for {material_id}")
        material["base_color"] = [float(np.clip(v, 0, 1)) for v in material["base_color"]]

    part_ids = set()
    for part in parts:
        part_id = str(part["id"])
        if not re.fullmatch(r"[a-z0-9_]{1,64}", part_id):
            raise ValueError(f"Unsafe part id: {part_id}")
        if part_id in part_ids:
            raise ValueError(f"Duplicate part id: {part_id}")
        part_ids.add(part_id)
        if part["primitive"] not in SECURITY.allowed_primitives:
            raise ValueError(f"Primitive not allowed: {part['primitive']}")
        for key in ("location", "rotation_deg", "scale"):
            if not _finite_triplet(part[key]):
                raise ValueError(f"Invalid {key} for {part_id}")
        part["scale"] = [float(np.clip(abs(v), 0.001, SECURITY.max_dimension_m)) for v in part["scale"]]
        if part["material_id"] not in material_ids:
            raise ValueError(f"Unknown material on {part_id}")
        for modifier in part.get("modifiers", []):
            if modifier["type"] not in SECURITY.allowed_modifiers:
                raise ValueError(f"Modifier not allowed: {modifier['type']}")
            modifier["strength"] = float(np.clip(modifier["strength"], 0, 10))

    for part in parts:
        parent_id = part.get("parent_id")
        if parent_id is not None and parent_id not in part_ids:
            raise ValueError(f"Unknown parent_id: {parent_id}")

    plan["lod_ratios"] = [
        float(np.clip(v, 0.01, 1.0)) for v in plan.get("lod_ratios", [1.0])
    ]
    plan["texture_policy"]["resolution"] = int(np.clip(
        plan["texture_policy"].get("resolution", 2048), 256, 4096
    ))
    return plan


def _select_live_planner_model(client) -> str:
    preferred = [PLANNER_MODEL] + list(PLANNER_FALLBACK_MODELS)
    available = {model.id for model in client.models.list().data}
    for model_id in preferred:
        if model_id in available:
            if model_id != PLANNER_MODEL:
                if not (ALLOW_PLANNER_FALLBACK and SECURITY.allow_model_fallback):
                    raise RuntimeError(
                        f"{PLANNER_MODEL} unavailable and fallback is disabled"
                    )
                print(f"Planner fallback: {PLANNER_MODEL} -> {model_id}")
            return model_id
    raise RuntimeError(
        f"None of the configured planner models are available: {preferred}"
    )


def plan_asset(prompt: str) -> tuple[dict[str, Any], str]:
    fallback = local_asset_plan(prompt)
    if not USE_GPT_PLANNER:
        return validate_asset_graph(fallback), "local_secure_fallback"

    key = get_openai_key() if "get_openai_key" in globals() else os.environ.get("OPENAI_API_KEY", "")
    if not key:
        return validate_asset_graph(fallback), "local_secure_fallback"

    try:
        from openai import OpenAI
        client = OpenAI(api_key=key)
        model_id = _select_live_planner_model(client)
        response = client.responses.create(
            model=model_id,
            input=[
                {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": (
                        "Plan this game asset:\n\n" + prompt +
                        "\n\nApproved primitives: " + ", ".join(SECURITY.allowed_primitives) +
                        "\nApproved modifiers: " + ", ".join(SECURITY.allowed_modifiers)
                    ),
                },
            ],
            text={
                "format": {
                    "type": "json_schema",
                    "name": "universal_asset_graph",
                    "description": "A secure bounded game-asset compilation plan",
                    "schema": ASSET_GRAPH_SCHEMA,
                    "strict": True,
                }
            },
        )
        parsed = json.loads(response.output_text)
        return validate_asset_graph(parsed), model_id
    except Exception as exc:
        print("GPT planner failed; using secure deterministic fallback:", repr(exc))
        return validate_asset_graph(fallback), "local_secure_fallback"


# get_openai_key is defined again in the texture cell, but make it available now.
def get_openai_key():
    key = os.environ.get("OPENAI_API_KEY", "")
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY") or ""
    except Exception:
        key = ""
    if not key:
        key = getpass("OPENAI_API_KEY (Enter for secure local fallback): ").strip()
    return key


ASSET_PLAN, PLANNER_USED = plan_asset(ASSET_PROMPT)
IS_TREE = ASSET_PLAN["strategy"] == "tree_polyflow"

(ROOT / "asset_graph.json").write_text(
    json.dumps(ASSET_PLAN, indent=2),
    encoding="utf-8",
)
print("Planner:", PLANNER_USED)
print("Strategy:", ASSET_PLAN["strategy"])
print("Category:", ASSET_PLAN["category"])
print("Asset ID:", ASSET_PLAN["asset_id"])
print("Materials:", len(ASSET_PLAN["materials"]))
print("Parts:", len(ASSET_PLAN["parts"]))


## 2. Procedural botanical skeleton and mesh

The skeleton uses four interacting mechanisms:

- **Phyllotactic branch launches** distribute primary branches around the trunk.
- **Space colonization** lets branch tips grow toward a crown attractor field.
- **Phototropism and inertia** keep growth upward and visually coherent.
- **Leonardo-style pipe radii** enforce approximately conserved cross-sectional capacity:

\[
r_p^\gamma \approx r_{tip}^{\gamma}+\sum_{c\in children(p)}r_c^\gamma,\qquad \gamma\approx 2.25.
\]

A parallel-transported local frame creates one ring per skeleton node, so the resulting surface stays connected across long branches while retaining predictable UV coordinates.


In [ ]:
if IS_TREE:
    import numpy as np
    from dataclasses import dataclass, asdict
    from scipy.spatial import cKDTree
    from scipy import sparse
    from scipy.sparse.linalg import eigsh
    import json, math, os

    @dataclass
    class TreeConfig:
        seed:int=42
        height:float=8.0
        trunk_nodes:int=22
        branch_levels:int=11
        branches_per_level:int=2
        attractor_count:int=1000
        growth_iterations:int=52
        influence_radius:float=1.6
        kill_radius:float=0.32
        step_size:float=0.24
        radial_segments:int=10
        base_radius:float=0.48
        tip_radius:float=0.014
        crown_radius:float=3.4
        crown_height:float=4.8


    def normalize(v, eps=1e-9):
        v=np.asarray(v,float); n=np.linalg.norm(v)
        return v/(n+eps)


    def generate_skeleton(cfg:TreeConfig):
        rng=np.random.default_rng(cfg.seed)
        pos=[]; parent=[]; kind=[]; prev_dir=[]
        # trunk
        for i in range(cfg.trunk_nodes):
            z=cfg.height*i/(cfg.trunk_nodes-1)
            sway=np.array([0.14*np.sin(i*0.43), 0.10*np.sin(i*0.29+1.4), 0.0])*(i/(cfg.trunk_nodes-1))
            p=np.array([0.,0.,z])+sway
            pos.append(p); parent.append(i-1 if i>0 else -1); kind.append('trunk')
            prev_dir.append(normalize(p-pos[i-1]) if i>0 else np.array([0.,0.,1.]))
        # roots
        for k in range(8):
            angle=2*np.pi*k/8 + rng.normal(0,0.12)
            pidx=0
            d=normalize([np.cos(angle),np.sin(angle),-0.18-rng.uniform(0,0.08)])
            for s in range(4+rng.integers(0,3)):
                p=pos[pidx]+d*(0.45*(0.88**s))
                p[2]=max(-0.25,p[2])
                pos.append(p); parent.append(pidx); kind.append('root'); prev_dir.append(d)
                pidx=len(pos)-1
                d=normalize(d+rng.normal(0,0.09,3)+np.array([0,0,-0.03]))
        # crown attractors in ellipsoid
        center=np.array([0.,0.,cfg.height*0.72])
        attrs=[]
        while len(attrs)<cfg.attractor_count:
            q=rng.uniform(-1,1,3)
            if np.dot(q,q)<=1:
                p=center+q*np.array([cfg.crown_radius,cfg.crown_radius,cfg.crown_height/2])
                # carve trunk cavity and favor upper crown
                if np.linalg.norm(p[:2])>0.55 and p[2]>cfg.height*0.33:
                    attrs.append(p)
        attrs=np.asarray(attrs)
        # seed branches from trunk
        active=[]
        levels=np.linspace(int(cfg.trunk_nodes*0.38),cfg.trunk_nodes-3,cfg.branch_levels).astype(int)
        for li,ti in enumerate(levels):
            n=cfg.branches_per_level + (1 if li%3==0 else 0)
            for b in range(n):
                ang=(2*np.pi*(b/n)+li*2.399963+rng.normal(0,0.18))%(2*np.pi)
                upward=0.25+0.45*(ti/(cfg.trunk_nodes-1))
                d=normalize([np.cos(ang),np.sin(ang),upward])
                p=np.asarray(pos[ti])+d*(cfg.step_size*1.25)
                pos.append(p); parent.append(int(ti)); kind.append('branch'); prev_dir.append(d)
                active.append(len(pos)-1)
        # grow branches with space colonization
        for it in range(cfg.growth_iterations):
            if len(attrs)==0 or len(active)==0: break
            active_pts=np.asarray([pos[i] for i in active])
            tree=cKDTree(active_pts)
            dist, idx=tree.query(attrs,k=1)
            # remove consumed attractors near any active tip (and occasionally all nodes)
            keep=dist>cfg.kill_radius
            attrs=attrs[keep]; dist=dist[keep]; idx=idx[keep]
            assignments={}
            for ai,(dd,ii) in enumerate(zip(dist,idx)):
                if dd<cfg.influence_radius:
                    assignments.setdefault(int(ii),[]).append(ai)
            new_active=[]
            for local_i,tip in enumerate(active):
                ids=assignments.get(local_i)
                if not ids:
                    continue
                vecs=attrs[ids]-np.asarray(pos[tip])
                vecs=np.array([normalize(v) for v in vecs])
                d=normalize(0.68*vecs.mean(axis=0)+0.20*prev_dir[tip]+np.array([0,0,0.12])+rng.normal(0,0.055,3))
                step=cfg.step_size*rng.uniform(0.86,1.12)
                p=np.asarray(pos[tip])+d*step
                # collision-ish repulsion from trunk centerline
                if np.linalg.norm(p[:2])<0.25 and p[2]>1.5:
                    p[:2]+=normalize(p[:2]+1e-3)[:2]*0.15
                pos.append(p); parent.append(tip); kind.append('branch'); prev_dir.append(d)
                new_active.append(len(pos)-1)
            active=new_active
        return np.asarray(pos,float), np.asarray(parent,int), kind


    def children_from_parent(parent):
        ch=[[] for _ in parent]
        for i,p in enumerate(parent):
            if p>=0: ch[p].append(i)
        return ch


    def compute_radii(pos,parent,cfg):
        ch=children_from_parent(parent)
        gamma=2.25
        r=np.full(len(pos),cfg.tip_radius,float)
        for i in range(len(pos)-1,-1,-1):
            if ch[i]:
                r[i]=(cfg.tip_radius**gamma+sum(r[c]**gamma for c in ch[i]))**(1/gamma)
        r*=cfg.base_radius/max(r[0],1e-6)
        # sensible minimum and root taper
        r=np.clip(r,cfg.tip_radius,cfg.base_radius)
        return r,ch


    def rodrigues_rotate(v,axis,angle):
        axis=normalize(axis)
        return v*np.cos(angle)+np.cross(axis,v)*np.sin(angle)+axis*np.dot(axis,v)*(1-np.cos(angle))


    def build_frames(pos,parent,ch):
        n=len(pos)
        tang=np.zeros((n,3))
        for i in range(n):
            dirs=[]
            if parent[i]>=0: dirs.append(normalize(pos[i]-pos[parent[i]]))
            for c in ch[i]: dirs.append(normalize(pos[c]-pos[i]))
            tang[i]=normalize(np.mean(dirs,axis=0) if dirs else np.array([0,0,1.]))
        u=np.zeros((n,3)); v=np.zeros((n,3))
        ref=np.array([1.,0.,0.]) if abs(tang[0][0])<0.9 else np.array([0.,1.,0.])
        u[0]=normalize(np.cross(tang[0],ref)); v[0]=normalize(np.cross(tang[0],u[0]))
        for i in range(1,n):
            p=parent[i]
            if p<0:
                ref=np.array([1.,0.,0.]) if abs(tang[i][0])<0.9 else np.array([0.,1.,0.])
                u[i]=normalize(np.cross(tang[i],ref)); v[i]=normalize(np.cross(tang[i],u[i])); continue
            a=tang[p]; b=tang[i]
            axis=np.cross(a,b); s=np.linalg.norm(axis); c=np.clip(np.dot(a,b),-1,1)
            if s<1e-6:
                ui=u[p]
            else:
                ui=rodrigues_rotate(u[p],axis/s,np.arctan2(s,c))
            ui=normalize(ui-b*np.dot(ui,b))
            if np.linalg.norm(ui)<1e-6:
                ref=np.array([1.,0.,0.]) if abs(b[0])<0.9 else np.array([0.,1.,0.])
                ui=normalize(np.cross(b,ref))
            u[i]=ui; v[i]=normalize(np.cross(b,ui))
        return tang,u,v


    def build_tree_mesh(pos,parent,radii,ch,cfg):
        rng=np.random.default_rng(cfg.seed+11)
        tang,u,v=build_frames(pos,parent,ch)
        nrad=cfg.radial_segments
        verts=[]; uvs=[]; node_ids=[]
        geod=np.zeros(len(pos))
        for i,p in enumerate(parent):
            if p>=0: geod[i]=geod[p]+np.linalg.norm(pos[i]-pos[p])
        for i,c in enumerate(pos):
            for k in range(nrad):
                a=2*np.pi*k/nrad
                rough=1.0+0.035*np.sin(7*a+0.7*i)+rng.normal(0,0.008)
                q=c+radii[i]*rough*(np.cos(a)*u[i]+np.sin(a)*v[i])
                verts.append(q); uvs.append([k/nrad,geod[i]*0.42]); node_ids.append(i)
        faces=[]
        for i,p in enumerate(parent):
            if p<0: continue
            for k in range(nrad):
                a=p*nrad+k; b=p*nrad+(k+1)%nrad; c=i*nrad+(k+1)%nrad; d=i*nrad+k
                faces.append([a,b,c]); faces.append([a,c,d])
        # caps at base and terminal tips
        terminals=[i for i in range(len(pos)) if not ch[i]]
        # base cap center
        base_center=len(verts); verts.append(pos[0]); uvs.append([0.5,0.5]); node_ids.append(0)
        for k in range(nrad): faces.append([base_center,(k+1)%nrad,k])
        for i in terminals:
            center=len(verts); verts.append(pos[i]); uvs.append([0.5,0.5]); node_ids.append(i)
            for k in range(nrad):
                a=i*nrad+k; b=i*nrad+(k+1)%nrad
                faces.append([center,a,b])
        return np.asarray(verts,float),np.asarray(faces,int),np.asarray(uvs,float),np.asarray(node_ids,int),geod,tang


    def select_bones(pos,parent,ch,kind,max_bones=240):
        selected={0}
        # trunk and junctions/tips
        for i in range(1,len(pos)):
            if kind[i]=='trunk' or len(ch[i])!=1 or i%4==0:
                selected.add(i)
        # reduce if needed, retain semantic nodes
        if len(selected)>max_bones:
            mandatory={0}|{i for i in range(len(pos)) if kind[i]=='trunk' or len(ch[i])!=1}
            optional=sorted(selected-mandatory)
            stride=max(1,math.ceil(len(optional)/max(1,max_bones-len(mandatory))))
            selected=mandatory|set(optional[::stride])
        selected=sorted(selected)
        selset=set(selected)
        nearest=np.zeros(len(pos),int)
        for i in range(len(pos)):
            j=i
            while j not in selset and parent[j]>=0: j=parent[j]
            nearest[i]=j if j in selset else 0
        bone_index={n:i for i,n in enumerate(selected)}
        bones=[]
        for n in selected:
            if n==0:
                psel=-1; head=pos[0]; tail=pos[1] if len(pos)>1 else pos[0]+[0,0,0.2]
            else:
                j=parent[n]
                while j not in selset and j>=0: j=parent[j]
                psel=j if j>=0 else 0
                head=pos[psel]; tail=pos[n]
                if np.linalg.norm(tail-head)<1e-4: tail=head+np.array([0,0,0.05])
            bones.append({'name':f'B_{bone_index[n]:03d}','node':int(n),'parent_node':int(psel),'parent_name':None if psel<0 else f'B_{bone_index[psel]:03d}','head':head.tolist(),'tail':tail.tolist()})
        return bones,nearest,bone_index


    def skin_weights(node_ids,parent,nearest,bone_index):
        indices=np.full((len(node_ids),4),-1,int); weights=np.zeros((len(node_ids),4),float)
        for vi,n in enumerate(node_ids):
            a=nearest[n]
            p=parent[n]
            b=nearest[p] if p>=0 else a
            if a==b:
                indices[vi,0]=bone_index[a]; weights[vi,0]=1.0
            else:
                indices[vi,0]=bone_index[a]; indices[vi,1]=bone_index[b]
                weights[vi,0]=0.72; weights[vi,1]=0.28
        return indices,weights


    def spectral_topology_embedding(vertices,faces,dim=32):
        n=len(vertices)
        I=[];J=[]
        for tri in faces:
            for a,b in ((tri[0],tri[1]),(tri[1],tri[2]),(tri[2],tri[0])):
                I.extend([a,b]);J.extend([b,a])
        A=sparse.coo_matrix((np.ones(len(I)),(I,J)),shape=(n,n)).tocsr(); A.data[:]=1
        deg=np.asarray(A.sum(axis=1)).ravel(); dinv=1/np.sqrt(np.maximum(deg,1e-8))
        L=sparse.eye(n)-sparse.diags(dinv)@A@sparse.diags(dinv)
        k=min(dim//2+1,n-2)
        vals,vecs=eigsh(L,k=k,which='SM')
        order=np.argsort(vals); vals=vals[order]; vecs=vecs[:,order]
        space=vecs[:,1:k]
        # pad to 16
        target=dim//2
        if space.shape[1]<target: space=np.pad(space,((0,0),(0,target-space.shape[1])))
        space=space[:,:target]
        # signed diffusion channels encode multi-scale topology progression
        scales=np.geomspace(0.5,8.0,target)
        lam=np.pad(vals[1:1+target],(0,max(0,target-(len(vals)-1))),constant_values=1.0)[:target]
        time=space*np.exp(-lam[None,:]*scales[None,:])
        def normcols(x):
            return (x-x.mean(0,keepdims=True))/(x.std(0,keepdims=True)+1e-6)
        emb=np.concatenate([normcols(space),normcols(time)],axis=1).astype(np.float32)
        return emb,A


    def leaf_positions(pos,ch,radii,cfg):
        rng=np.random.default_rng(cfg.seed+31)
        candidates=[i for i in range(len(pos)) if (not ch[i] and pos[i,2]>cfg.height*0.35) or (radii[i]<0.045 and pos[i,2]>cfg.height*0.45)]
        if not candidates: return np.empty((0,3)),np.empty((0,),int)
        count=min(520,max(180,len(candidates)*3))
        ids=rng.choice(candidates,size=count,replace=True)
        offs=rng.normal(size=(count,3)); offs[:,2]*=0.35
        offs=np.array([normalize(o) for o in offs])*rng.uniform(0.05,0.25,(count,1))
        return pos[ids]+offs,ids


else:
    print('Skipped tree-only stage: def normalize(')


In [ ]:
if IS_TREE:
    #@title Generate tree skeleton, surface mesh, leaves, and initial rig
    positions, parents, kinds = generate_skeleton(CFG)
    radii, children = compute_radii(positions, parents, CFG)
    vertices, faces, uvs, vertex_node_ids, node_geodesic, node_tangents = build_tree_mesh(
        positions, parents, radii, children, CFG
    )

    bones, nearest_selected_node, bone_index = select_bones(
        positions, parents, children, kinds, max_bones=CFG.max_bones
    )
    skin_indices, skin_weights_array = skin_weights(
        vertex_node_ids, parents, nearest_selected_node, bone_index
    )
    leaf_xyz, leaf_node_ids = leaf_positions(positions, children, radii, CFG)
    if len(leaf_xyz) > CFG.max_leaves:
        leaf_xyz = leaf_xyz[:CFG.max_leaves]
        leaf_node_ids = leaf_node_ids[:CFG.max_leaves]

    np.savez_compressed(
        ROOT / 'skeleton.npz',
        positions=positions,
        parents=parents,
        radii=radii,
        node_geodesic=node_geodesic,
        node_tangents=node_tangents,
        leaf_xyz=leaf_xyz,
        leaf_node_ids=leaf_node_ids,
    )
    np.savez_compressed(
        ROOT / 'tree_mesh.npz',
        vertices=vertices,
        faces=faces,
        uvs=uvs,
        vertex_node_ids=vertex_node_ids,
        skin_indices=skin_indices,
        skin_weights=skin_weights_array,
    )

    # Blender installed through apt uses its own isolated Python runtime. It may
    # not contain NumPy even though the Colab kernel does. Write a compressed
    # standard-library bridge so Blender only needs gzip and json.
    import gzip

    skeleton_bridge = {
        'positions': positions.tolist(),
        'parents': parents.tolist(),
        'radii': radii.tolist(),
        'node_geodesic': node_geodesic.tolist(),
        'node_tangents': node_tangents.tolist(),
        'leaf_xyz': leaf_xyz.tolist(),
        'leaf_node_ids': leaf_node_ids.tolist(),
    }
    mesh_bridge = {
        'vertices': vertices.tolist(),
        'faces': faces.tolist(),
        'uvs': uvs.tolist(),
        'vertex_node_ids': vertex_node_ids.tolist(),
        'skin_indices': skin_indices.tolist(),
        'skin_weights': skin_weights_array.tolist(),
    }

    with gzip.open(ROOT / 'skeleton.json.gz', 'wt', encoding='utf-8') as handle:
        json.dump(skeleton_bridge, handle, separators=(',', ':'))
    with gzip.open(ROOT / 'tree_mesh.json.gz', 'wt', encoding='utf-8') as handle:
        json.dump(mesh_bridge, handle, separators=(',', ':'))

    rig_payload = {
        'bones': bones,
        'bone_names': [b['name'] for b in bones],
        'source': 'PolyFlow-inspired procedural rig map',
    }
    (ROOT / 'rig_map.json').write_text(json.dumps(rig_payload, indent=2))

    print(f'Skeleton nodes: {len(positions):,}')
    print(f'Mesh vertices:   {len(vertices):,}')
    print(f'Mesh triangles:  {len(faces):,}')
    print(f'Bones:           {len(bones):,}')
    print(f'Leaf cards:      {len(leaf_xyz):,}')
    print('Blender bridge:', ROOT / 'tree_mesh.json.gz')
    print('Blender bridge:', ROOT / 'skeleton.json.gz')

else:
    print('Skipped tree-only stage: #@title Generate tree skeleton, surface mesh, leaves, and initial rig')


In [ ]:
if IS_TREE:
    #@title Preview the generated skeleton and crown
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    for i, p in enumerate(parents):
        if p >= 0:
            seg = positions[[p, i]]
            width = max(0.25, radii[p] * 5.0)
            ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], linewidth=width)

    if len(leaf_xyz):
        ax.scatter(leaf_xyz[:, 0], leaf_xyz[:, 1], leaf_xyz[:, 2], s=2, alpha=0.35)

    ax.set_box_aspect((1, 1, 1.25))
    ax.set_title('Procedural skeleton, pipe radii, and leaf distribution')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    plt.show()

else:
    print('Skipped tree-only stage: #@title Preview the generated skeleton and crown')


## 3. PolyFlow-inspired continuous topology and rigging-surface comprehension

PolyFlow represents each vertex using a continuous state

\[
\mathbf{z}_i=[\mathbf{p}_i,\mathbf{n}_i,\mathbf{e}_i],
\]

where `e_i` is a continuous topology embedding. Adjacency can be decoded with a spacetime metric

\[
d^{st}(\mathbf{e}_i,\mathbf{e}_j)
=\|\mathbf{e}_i^s-\mathbf{e}_j^s\|_2^2
-\|\mathbf{e}_i^t-\mathbf{e}_j^t\|_2^2.
\]

This notebook computes a **spectral-diffusion surrogate** for the 32D topology field, then augments it into a rig-aware state:

\[
\mathbf{q}_i=
[\mathbf{p}_i,\mathbf{n}_i,\mathbf{e}_i,
 g_i,\rho_i,\kappa_i,s_i,\mathbf{w}_i].
\]

Here `g` is root geodesic distance, `ρ` is branch radius, `κ` is a curvature proxy, `s` is stiffness, and `w` contains sparse bone influences. This makes the mapping useful for animation, wind, cutting, damage, LOD transfer, and neural deformation.


In [ ]:
if IS_TREE:
    #@title Surface-feature helpers and topology-threshold calibration
    def compute_vertex_normals(vertices: np.ndarray, faces: np.ndarray) -> np.ndarray:
        normals = np.zeros_like(vertices, dtype=np.float64)
        tri = vertices[faces]
        fn = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
        area = np.linalg.norm(fn, axis=1, keepdims=True)
        fn = fn / np.maximum(area, 1e-12)
        for corner in range(3):
            np.add.at(normals, faces[:, corner], fn)
        normals /= np.maximum(np.linalg.norm(normals, axis=1, keepdims=True), 1e-12)
        return normals.astype(np.float32)


    def sparse_edges_from_faces(faces: np.ndarray, vertex_count: int):
        ii, jj = [], []
        for tri in faces:
            for a, b in ((tri[0], tri[1]), (tri[1], tri[2]), (tri[2], tri[0])):
                ii.extend([int(a), int(b)])
                jj.extend([int(b), int(a)])
        A = sparse.coo_matrix((np.ones(len(ii)), (ii, jj)), shape=(vertex_count, vertex_count)).tocsr()
        A.data[:] = 1.0
        A.eliminate_zeros()
        return A


    def calibrate_spacetime_threshold(embedding, adjacency, seed=42, samples=25000):
        rng = np.random.default_rng(seed)
        upper = sparse.triu(adjacency, k=1).tocoo()
        edge_pairs = np.stack([upper.row, upper.col], axis=1)
        if len(edge_pairs) > samples:
            edge_pairs = edge_pairs[rng.choice(len(edge_pairs), samples, replace=False)]

        edge_set = set(map(tuple, edge_pairs.tolist()))
        nonedge = []
        n = embedding.shape[0]
        while len(nonedge) < len(edge_pairs):
            a = int(rng.integers(0, n)); b = int(rng.integers(0, n))
            if a == b: continue
            if a > b: a, b = b, a
            if (a, b) not in edge_set and adjacency[a, b] == 0:
                nonedge.append((a, b))
        nonedge = np.asarray(nonedge, dtype=np.int64)

        ds = embedding.shape[1] // 2
        def distance(pairs):
            d = embedding[pairs[:, 0]] - embedding[pairs[:, 1]]
            return np.sum(d[:, :ds] ** 2, axis=1) - np.sum(d[:, ds:] ** 2, axis=1)

        de = distance(edge_pairs)
        dn = distance(nonedge)
        all_d = np.concatenate([de, dn])
        y = np.concatenate([np.ones_like(de, dtype=np.int32), np.zeros_like(dn, dtype=np.int32)])

        best = {'f1': -1.0, 'tau': 0.0, 'precision': 0.0, 'recall': 0.0}
        for tau in np.quantile(all_d, np.linspace(0.01, 0.99, 300)):
            pred = all_d < tau
            tp = int(np.sum(pred & (y == 1)))
            fp = int(np.sum(pred & (y == 0)))
            fn = int(np.sum((~pred) & (y == 1)))
            precision = tp / max(tp + fp, 1)
            recall = tp / max(tp + fn, 1)
            f1 = 2 * precision * recall / max(precision + recall, 1e-12)
            if f1 > best['f1']:
                best = {'f1': float(f1), 'tau': float(tau), 'precision': float(precision), 'recall': float(recall)}
        return best


    def strahler_orders(children):
        order = np.ones(len(children), dtype=np.int32)
        for i in range(len(children) - 1, -1, -1):
            if not children[i]:
                order[i] = 1
            else:
                vals = sorted([order[c] for c in children[i]], reverse=True)
                order[i] = vals[0] + 1 if len(vals) > 1 and vals[0] == vals[1] else vals[0]
        return order


    def node_curvature(positions, parents, children):
        kappa = np.zeros(len(positions), dtype=np.float32)
        for i in range(len(positions)):
            if parents[i] < 0 or not children[i]:
                continue
            incoming = normalize(positions[i] - positions[parents[i]])
            outgoing = normalize(np.mean([normalize(positions[c] - positions[i]) for c in children[i]], axis=0))
            kappa[i] = float(np.arccos(np.clip(np.dot(incoming, outgoing), -1.0, 1.0)))
        return kappa

else:
    print('Skipped tree-only stage: #@title Surface-feature helpers and topology-threshold calibration')


In [ ]:
if IS_TREE:
    #@title Compute topology embedding, rig descriptors, and a complete vertex state
    vertex_normals = compute_vertex_normals(vertices, faces)
    topology_embedding, adjacency = spectral_topology_embedding(vertices, faces, dim=32)
    threshold_report = calibrate_spacetime_threshold(topology_embedding, adjacency, seed=CFG.seed)

    orders = strahler_orders(children)
    node_kappa = node_curvature(positions, parents, children)
    vertex_radius = radii[vertex_node_ids].astype(np.float32)
    vertex_geodesic = node_geodesic[vertex_node_ids].astype(np.float32)
    vertex_curvature = node_kappa[vertex_node_ids].astype(np.float32)
    vertex_order = orders[vertex_node_ids].astype(np.float32)

    # Simplified cantilever stiffness proxy: E*I/L, normalized for learning/visualization.
    # The absolute Young's modulus is intentionally omitted; this is a relative game-physics field.
    segment_length = np.ones(len(positions), dtype=np.float32)
    for i, p in enumerate(parents):
        if p >= 0:
            segment_length[i] = np.linalg.norm(positions[i] - positions[p])
    node_stiffness = (np.pi * radii**4 / 4.0) / np.maximum(segment_length, 1e-4)
    node_stiffness = node_stiffness / max(node_stiffness.max(), 1e-8)
    vertex_stiffness = node_stiffness[vertex_node_ids].astype(np.float32)

    surface_state = np.concatenate([
        vertices.astype(np.float32),
        vertex_normals,
        topology_embedding.astype(np.float32),
        vertex_geodesic[:, None],
        vertex_radius[:, None],
        vertex_curvature[:, None],
        vertex_stiffness[:, None],
        vertex_order[:, None],
    ], axis=1)

    np.savez_compressed(
        ROOT / 'surface_comprehension.npz',
        surface_state=surface_state,
        topology_embedding=topology_embedding,
        vertex_normals=vertex_normals,
        vertex_geodesic=vertex_geodesic,
        vertex_radius=vertex_radius,
        vertex_curvature=vertex_curvature,
        vertex_stiffness=vertex_stiffness,
        vertex_order=vertex_order,
        adjacency_indptr=adjacency.indptr,
        adjacency_indices=adjacency.indices,
        spacetime_tau=np.array([threshold_report['tau']], dtype=np.float32),
    )

    (ROOT / 'topology_diagnostics.json').write_text(json.dumps({
        'embedding_dimension': 32,
        'space_dimension': 16,
        'time_dimension': 16,
        'sampled_spacetime_decoder': threshold_report,
        'warning': 'This is a spectral-diffusion surrogate, not the trained PolyFlow topology embedder.',
    }, indent=2))

    print('Surface state shape:', surface_state.shape)
    print('Sampled spacetime decoder:', json.dumps(threshold_report, indent=2))

else:
    print('Skipped tree-only stage: #@title Compute topology embedding, rig descriptors, and a complete ve')


In [ ]:
if IS_TREE:
    #@title Visualize continuous surface semantics
    sample = np.linspace(0, len(vertices) - 1, min(12000, len(vertices))).astype(int)
    fig = plt.figure(figsize=(11, 9))
    ax = fig.add_subplot(111, projection='3d')
    sc = ax.scatter(
        vertices[sample, 0], vertices[sample, 1], vertices[sample, 2],
        c=vertex_geodesic[sample], s=1.3
    )
    ax.set_title('Rigging-surface comprehension: root geodesic field')
    ax.set_box_aspect((1, 1, 1.25))
    fig.colorbar(sc, ax=ax, shrink=0.65, label='Geodesic distance from root')
    plt.show()

else:
    print('Skipped tree-only stage: #@title Visualize continuous surface semantics')


## 4. GPT Image texture synthesis with procedural fallback

The texture prompts request **flat, tileable material data**, not a rendered tree. GPT Image creates the bark albedo and a transparent leaf card. The notebook then derives roughness and normal maps with deterministic image processing.

For Colab secrets, add `OPENAI_API_KEY` under the key icon in the left sidebar. You may also paste it when prompted. The key is never written to the notebook outputs.


In [ ]:
#@title Universal GPT Image texture foundry with physically scaled foliage profiles
try:
    from openai import OpenAI
except Exception as _openai_import_error:
    OpenAI = None
    print("OpenAI unavailable; procedural textures will be used:", repr(_openai_import_error))


def save_b64_image(result, path: Path):
    raw = base64.b64decode(result.data[0].b64_json, validate=True)
    if len(raw) > SECURITY.max_api_image_bytes:
        raise ValueError("Refusing oversized API image payload")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(raw)
    decoded = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if decoded is None:
        path.unlink(missing_ok=True)
        raise ValueError("API payload is not a supported image")


def _write_png(path: Path, image: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    array = np.asarray(image)
    if array.dtype != np.uint8:
        array = np.uint8(np.clip(array, 0, 255))
    if not cv2.imwrite(str(path), array):
        raise IOError(f"OpenCV failed to write {path}")


def _safe_texture_name(value: str) -> str:
    cleaned = re.sub(r"[^a-z0-9_]+", "_", value.lower()).strip("_")
    if not re.fullmatch(r"[a-z0-9_]{1,64}", cleaned):
        raise ValueError(f"Unsafe texture name: {value}")
    return cleaned


def procedural_material(path: Path, base_color, metallic, roughness, seed=1, transparent=False):
    size = 1024
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size]
    coarse = rng.normal(0, 1, (size, size)).astype(np.float32)
    coarse = cv2.GaussianBlur(coarse, (0, 0), 18)
    coarse /= max(float(np.std(coarse)), 1e-6)
    fine = rng.normal(0, 1, (size, size)).astype(np.float32)
    fine = cv2.GaussianBlur(fine, (0, 0), 2.2)
    fine /= max(float(np.std(fine)), 1e-6)
    wave = np.sin(xx / 41.0 + 0.6 * np.sin(yy / 73.0))
    variation = np.clip(0.12 * coarse + 0.045 * fine + 0.025 * wave, -0.22, 0.22)

    rgb = np.zeros((size, size, 3), dtype=np.float32)
    base = np.asarray(base_color, dtype=np.float32)
    for channel in range(3):
        rgb[..., channel] = np.clip(base[channel] + variation, 0, 1)

    bgr = np.uint8(rgb[..., ::-1] * 255)
    if transparent:
        alpha = np.uint8(np.clip(0.78 + 0.22 * coarse, 0, 1) * 255)
        bgra = np.dstack([bgr, alpha])
        _write_png(path, bgra)
    else:
        _write_png(path, bgr)


def procedural_leaf_profile(path: Path, profile: dict, seed=73):
    size = 1024
    rng = np.random.default_rng(seed)
    rgba = np.zeros((size, size, 4), dtype=np.uint8)
    cx, cy = size // 2, size // 2
    profile_id = profile["id"]
    physical = float(profile["physical_length_m"])

    if profile_id == "cluster":
        leaf_count = 5
        for leaf_index in range(leaf_count):
            angle = (leaf_index / leaf_count) * 2 * np.pi + rng.uniform(-0.18, 0.18)
            center = (
                int(cx + np.cos(angle) * size * 0.12),
                int(cy + np.sin(angle) * size * 0.12),
            )
            _draw_leaf_shape(
                rgba, center, angle, 0.25 * size,
                color_shift=leaf_index * 4, damaged=False,
            )
    else:
        damaged = profile_id == "damaged"
        scale_map = {
            "micro": 0.27,
            "small": 0.31,
            "medium": 0.36,
            "large": 0.41,
            "damaged": 0.36,
        }
        _draw_leaf_shape(
            rgba, (cx, cy), 0.0,
            scale_map.get(profile_id, 0.36) * size,
            color_shift=int(physical * 100),
            damaged=damaged,
        )
    _write_png(path, rgba)


def _draw_leaf_shape(canvas, center, angle, length_px, color_shift=0, damaged=False):
    cx, cy = center
    width_px = length_px * 0.38
    points = []
    serration_count = 18 if not damaged else 13
    for i in range(260):
        t = 2 * np.pi * i / 260
        longitudinal = np.cos(t)
        width = np.sin(t) * (0.72 + 0.28 * (1.0 - abs(longitudinal)))
        serration = 1.0 + 0.035 * np.sin(t * serration_count)
        local_x = longitudinal * length_px
        local_y = width * width_px * serration
        x = cx + int(np.cos(angle) * local_x - np.sin(angle) * local_y)
        y = cy + int(np.sin(angle) * local_x + np.cos(angle) * local_y)
        points.append((x, y))

    polygon = np.asarray(points, dtype=np.int32)
    color = (
        int(np.clip(42 + color_shift, 28, 82)),
        int(np.clip(112 + color_shift, 82, 164)),
        int(np.clip(48 + color_shift // 2, 34, 96)),
        255,
    )
    cv2.fillPoly(canvas, [polygon], color=color, lineType=cv2.LINE_AA)

    start = (
        int(cx - np.cos(angle) * length_px),
        int(cy - np.sin(angle) * length_px),
    )
    end = (
        int(cx + np.cos(angle) * length_px),
        int(cy + np.sin(angle) * length_px),
    )
    cv2.line(canvas, start, end, (60, 148, 92, 255), max(2, int(length_px / 90)), cv2.LINE_AA)

    if damaged:
        for dx, dy, radius in [(-0.08, 0.05, 0.035), (0.12, -0.06, 0.025), (0.20, 0.07, 0.022)]:
            hole_center = (
                int(cx + dx * length_px * 2),
                int(cy + dy * length_px * 2),
            )
            cv2.circle(canvas, hole_center, max(4, int(radius * length_px)), (0, 0, 0, 0), -1, cv2.LINE_AA)


def _material_prompt(material: dict, variant_index: int) -> str:
    transparency = (
        "transparent background with isolated material element"
        if material["transparent"]
        else "fully opaque seamless square material scan"
    )
    return (
        f"Create variant {variant_index + 1} of a physically based game material. "
        f"Subject: {material['description']}. {ASSET_PLAN['art_direction']}. "
        f"{transparency}. Neutral orthographic material capture, no perspective, "
        "no directional lighting, no cast shadows, no baked highlights, no text, "
        "no logos, coherent real-world scale, clean edges, production-quality albedo."
    )


def _generate_image(client, prompt: str, path: Path, transparent: bool):
    kwargs = {
        "model": IMAGE_MODEL,
        "prompt": prompt,
        "size": "1024x1024",
        "quality": "high",
        "output_format": "png",
    }
    if transparent:
        kwargs["background"] = "transparent"
    result = client.images.generate(**kwargs)
    save_b64_image(result, path)


def generate_universal_texture_set():
    texture_root = ROOT / "textures"
    texture_root.mkdir(parents=True, exist_ok=True)
    key = get_openai_key()
    client = OpenAI(api_key=key) if key and OpenAI is not None and USE_GPT_IMAGES else None
    manifest = {
        "version": 2,
        "asset_id": ASSET_PLAN["asset_id"],
        "image_model": IMAGE_MODEL if client else "procedural_fallback",
        "materials": [],
        "leaf_profiles": [],
    }

    for material_index, material in enumerate(ASSET_PLAN["materials"]):
        material_id = _safe_texture_name(material["id"])
        record = {"id": material_id, "variants": []}
        for variant_index in range(material["texture_variants"]):
            output = texture_root / f"{material_id}_v{variant_index:02d}_albedo.png"
            generated = False
            if client is not None:
                try:
                    _generate_image(
                        client,
                        _material_prompt(material, variant_index),
                        output,
                        material["transparent"],
                    )
                    generated = True
                except Exception as exc:
                    print(f"GPT Image failed for {material_id} v{variant_index}:", repr(exc))
            if not generated:
                procedural_material(
                    output,
                    material["base_color"],
                    material["metallic"],
                    material["roughness"],
                    seed=1000 + material_index * 97 + variant_index,
                    transparent=material["transparent"],
                )
            record["variants"].append({
                "albedo": str(output.relative_to(ROOT)),
                "generated_by_gpt": generated,
            })
        manifest["materials"].append(record)

    if IS_TREE:
        for profile_index, profile in enumerate(ASSET_PLAN["texture_policy"]["leaf_profiles"]):
            profile_id = _safe_texture_name(profile["id"])
            output = texture_root / f"leaf_{profile_id}_albedo.png"
            generated = False
            if client is not None:
                try:
                    _generate_image(client, profile["prompt"], output, True)
                    generated = True
                except Exception as exc:
                    print(f"GPT leaf profile failed for {profile_id}:", repr(exc))
            if not generated:
                procedural_leaf_profile(output, profile, seed=5000 + profile_index * 131)
            manifest["leaf_profiles"].append({
                **profile,
                "albedo": str(output.relative_to(ROOT)),
                "generated_by_gpt": generated,
            })

        # Backward-compatible aliases for the PolyFlow tree compiler.
        bark_source = texture_root / "bark_v00_albedo.png"
        if not bark_source.exists():
            bark_source = texture_root / f"{ASSET_PLAN['materials'][0]['id']}_v00_albedo.png"
        shutil.copyfile(bark_source, ROOT / "bark_albedo.png")
        preferred_leaf = next(
            (
                ROOT / entry["albedo"]
                for entry in manifest["leaf_profiles"]
                if entry["id"] == "medium"
            ),
            ROOT / manifest["leaf_profiles"][0]["albedo"],
        )
        shutil.copyfile(preferred_leaf, ROOT / "leaf_albedo.png")

    (ROOT / "texture_manifest.json").write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )
    return manifest


TEXTURE_MANIFEST = generate_universal_texture_set()
print("Generated material families:", len(TEXTURE_MANIFEST["materials"]))
print("Generated leaf profiles:", len(TEXTURE_MANIFEST["leaf_profiles"]))


In [ ]:
#@title Preview generated material and multi-scale foliage textures
texture_paths = []
for material in TEXTURE_MANIFEST["materials"]:
    for variant in material["variants"]:
        texture_paths.append(ROOT / variant["albedo"])
for profile in TEXTURE_MANIFEST["leaf_profiles"]:
    texture_paths.append(ROOT / profile["albedo"])

print(f"Texture previews: {len(texture_paths)}")
for path in texture_paths[:16]:
    print(path.relative_to(ROOT))
    display(IPyImage(filename=str(path), width=240))


In [ ]:
#@title Derive universal PBR maps without Pillow
def mirror_tileable_array(image: np.ndarray) -> np.ndarray:
    arr = np.asarray(image)
    h, w = arr.shape[:2]
    lr = np.concatenate([arr, arr[:, ::-1]], axis=1)
    quad = np.concatenate([lr, lr[::-1]], axis=0)
    return np.ascontiguousarray(quad[h//2:h//2+h, w//2:w//2+w])


def derive_pbr_set(albedo_path: Path, prefix: Path, tileable=True, metallic=0.0, roughness_bias=0.6):
    source = cv2.imread(str(albedo_path), cv2.IMREAD_UNCHANGED)
    if source is None:
        raise FileNotFoundError(f"Could not decode {albedo_path}")

    alpha = source[..., 3] if source.ndim == 3 and source.shape[2] == 4 else None
    bgr = source[..., :3]
    if tileable:
        bgr = mirror_tileable_array(bgr)
        if alpha is not None:
            alpha = mirror_tileable_array(alpha)

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    gray = cv2.GaussianBlur(gray, (0, 0), 1.1)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    nx, ny, nz = -gx * 4.25, -gy * 4.25, np.ones_like(gray)
    norm = np.sqrt(nx * nx + ny * ny + nz * nz)
    normal_rgb = np.uint8(
        np.clip(np.stack([nx/norm, ny/norm, nz/norm], axis=-1) * 0.5 + 0.5, 0, 1) * 255
    )
    normal_bgr = normal_rgb[..., ::-1]
    rough = np.uint8(np.clip(roughness_bias + 0.28 * (1.0 - gray), 0, 1) * 255)
    ao = np.uint8(np.clip(0.72 + 0.28 * gray, 0, 1) * 255)
    metal = np.full_like(ao, int(np.clip(metallic, 0, 1) * 255))

    albedo_out = prefix.with_name(prefix.name + "_albedo.png")
    normal_out = prefix.with_name(prefix.name + "_normal.png")
    rough_out = prefix.with_name(prefix.name + "_roughness.png")
    ao_out = prefix.with_name(prefix.name + "_ao.png")
    metal_out = prefix.with_name(prefix.name + "_metallic.png")

    if alpha is not None:
        albedo_bgra = np.dstack([bgr, alpha])
        _write_png(albedo_out, albedo_bgra)
    else:
        _write_png(albedo_out, bgr)
    _write_png(normal_out, normal_bgr)
    _write_png(rough_out, rough)
    _write_png(ao_out, ao)
    _write_png(metal_out, metal)

    return {
        "albedo": str(albedo_out.relative_to(ROOT)),
        "normal": str(normal_out.relative_to(ROOT)),
        "roughness": str(rough_out.relative_to(ROOT)),
        "ao": str(ao_out.relative_to(ROOT)),
        "metallic": str(metal_out.relative_to(ROOT)),
    }


material_lookup = {m["id"]: m for m in ASSET_PLAN["materials"]}
for material_record in TEXTURE_MANIFEST["materials"]:
    material = material_lookup[material_record["id"]]
    for variant_index, variant in enumerate(material_record["variants"]):
        source = ROOT / variant["albedo"]
        prefix = ROOT / "textures" / f"{material['id']}_v{variant_index:02d}"
        variant["pbr"] = derive_pbr_set(
            source,
            prefix,
            tileable=not material["transparent"],
            metallic=material["metallic"],
            roughness_bias=material["roughness"],
        )

for profile in TEXTURE_MANIFEST["leaf_profiles"]:
    source = ROOT / profile["albedo"]
    prefix = ROOT / "textures" / f"leaf_{profile['id']}"
    profile["pbr"] = derive_pbr_set(
        source,
        prefix,
        tileable=False,
        metallic=0.0,
        roughness_bias=0.58,
    )

# Legacy tree aliases used by the existing specialized Blender compiler.
if IS_TREE:
    bark_variant = TEXTURE_MANIFEST["materials"][0]["variants"][0]["pbr"]
    shutil.copyfile(ROOT / bark_variant["albedo"], ROOT / "bark_albedo.png")
    shutil.copyfile(ROOT / bark_variant["normal"], ROOT / "bark_normal.png")
    shutil.copyfile(ROOT / bark_variant["roughness"], ROOT / "bark_roughness.png")

(ROOT / "texture_manifest.json").write_text(
    json.dumps(TEXTURE_MANIFEST, indent=2),
    encoding="utf-8",
)
print("Universal PBR derivation complete")


## 5. Blender armature, PBR material, render, and GLB export

The Blender stage:

- imports the connected trunk surface;
- creates one edit bone per selected skeleton control;
- adds sparse vertex groups and an Armature modifier;
- creates crossed foliage cards with alpha blending;
- builds bark albedo/normal/roughness materials;
- lights and renders the tree as a game-asset beauty shot;
- exports `tree_game_asset.glb`, `tree_asset.blend`, and `tree_render.png`.


## Blender compatibility bridge

Colab's Python and the apt-installed Blender Python are separate
environments. Blender 3.0.1 therefore may not see NumPy even when NumPy
works in the notebook.

This version exports mesh and skeleton arrays as compressed JSON and the
Blender builder uses only `gzip`, `json`, `random`, and `mathutils`.
Blender is launched with `--factory-startup` to avoid stale user color
management configuration.


In [ ]:
#@title Write the trusted Blender compiler selected by the AssetGraph
tree_blender_script = 'import bpy, json, os, math, gzip, random\nfrom mathutils import Vector\nfrom pathlib import Path\n\nROOT = Path(r"__ROOT__")\nRENDER_CFG = json.loads((ROOT / \'config.json\').read_text())[\'render\']\n\n# Clean scene\nbpy.ops.object.select_all(action=\'SELECT\')\nbpy.ops.object.delete(use_global=False)\nfor datablocks in (bpy.data.meshes, bpy.data.curves, bpy.data.materials, bpy.data.cameras, bpy.data.lights):\n    pass\n\ndef load_json_gz(path):\n    with gzip.open(path, \'rt\', encoding=\'utf-8\') as handle:\n        return json.load(handle)\n\nmesh_data = load_json_gz(ROOT / \'tree_mesh.json.gz\')\nskeleton_data = load_json_gz(ROOT / \'skeleton.json.gz\')\nrig = json.loads((ROOT / \'rig_map.json\').read_text())\ntexture_manifest = json.loads((ROOT / \'texture_manifest.json\').read_text())\n\nvertices = mesh_data[\'vertices\']\nfaces = mesh_data[\'faces\']\nuvs = mesh_data[\'uvs\']\nskin_indices = mesh_data[\'skin_indices\']\nskin_weights = mesh_data[\'skin_weights\']\nleaf_xyz = skeleton_data[\'leaf_xyz\']\nleaf_node_ids = skeleton_data[\'leaf_node_ids\']\nparents = skeleton_data[\'parents\']\n\n# ---------- Material helpers ----------\ndef safe_colorspace(image, requested=\'sRGB\'):\n    # Blender/OCIO builds differ. Blender 3.0.1 in Colab may expose only\n    # Linear and sRGB, while newer builds also expose Non-Color.\n    data_request = requested not in (\'sRGB\', \'Filmic sRGB\')\n    candidates = (\n        [\'Non-Color\', \'Raw\', \'Linear\', \'sRGB\']\n        if data_request else\n        [\'sRGB\', \'Filmic sRGB\', \'Linear\']\n    )\n    for candidate in candidates:\n        try:\n            image.colorspace_settings.name = candidate\n            return candidate\n        except (TypeError, ValueError):\n            continue\n    return image.colorspace_settings.name\n\ndef image_node(nodes, path, name, colorspace=\'sRGB\'):\n    tex = nodes.new(\'ShaderNodeTexImage\')\n    tex.name = name\n    tex.image = bpy.data.images.load(str(path), check_existing=True)\n    safe_colorspace(tex.image, colorspace)\n    tex.extension = \'REPEAT\'\n    return tex\n\nbark = bpy.data.materials.new(\'M_Bark_PBR\')\nbark.use_nodes = True\nbn = bark.node_tree.nodes\nbl = bark.node_tree.links\nfor node in list(bn):\n    bn.remove(node)\nout = bn.new(\'ShaderNodeOutputMaterial\')\nbsdf = bn.new(\'ShaderNodeBsdfPrincipled\')\nalbedo = image_node(bn, ROOT/\'bark_albedo.png\', \'Bark Albedo\', \'sRGB\')\nnormal_tex = image_node(bn, ROOT/\'bark_normal.png\', \'Bark Normal\', \'Non-Color\')\nrough_tex = image_node(bn, ROOT/\'bark_roughness.png\', \'Bark Roughness\', \'Non-Color\')\nnormal_map = bn.new(\'ShaderNodeNormalMap\')\nnormal_map.inputs[\'Strength\'].default_value = 0.85\nbl.new(albedo.outputs[\'Color\'], bsdf.inputs[\'Base Color\'])\nbl.new(normal_tex.outputs[\'Color\'], normal_map.inputs[\'Color\'])\nbl.new(normal_map.outputs[\'Normal\'], bsdf.inputs[\'Normal\'])\nbl.new(rough_tex.outputs[\'Color\'], bsdf.inputs[\'Roughness\'])\nif \'Specular IOR Level\' in bsdf.inputs:\n    bsdf.inputs[\'Specular IOR Level\'].default_value = 0.28\nelif \'Specular\' in bsdf.inputs:\n    bsdf.inputs[\'Specular\'].default_value = 0.28\nbl.new(bsdf.outputs[\'BSDF\'], out.inputs[\'Surface\'])\n\ndef make_leaf_material(profile):\n    profile_id = profile[\'id\']\n    material = bpy.data.materials.new(\'M_Leaf_\' + profile_id)\n    material.use_nodes = True\n    nodes = material.node_tree.nodes\n    links = material.node_tree.links\n    for node in list(nodes):\n        nodes.remove(node)\n    output = nodes.new(\'ShaderNodeOutputMaterial\')\n    bsdf_leaf = nodes.new(\'ShaderNodeBsdfPrincipled\')\n    pbr = profile.get(\'pbr\', {})\n    albedo_path = ROOT / pbr.get(\'albedo\', profile[\'albedo\'])\n    albedo_tex = image_node(nodes, albedo_path, \'Leaf \' + profile_id, \'sRGB\')\n    links.new(albedo_tex.outputs[\'Color\'], bsdf_leaf.inputs[\'Base Color\'])\n    links.new(albedo_tex.outputs[\'Alpha\'], bsdf_leaf.inputs[\'Alpha\'])\n    bsdf_leaf.inputs[\'Roughness\'].default_value = 0.62\n    if \'Subsurface Weight\' in bsdf_leaf.inputs:\n        bsdf_leaf.inputs[\'Subsurface Weight\'].default_value = 0.045\n    elif \'Subsurface\' in bsdf_leaf.inputs:\n        bsdf_leaf.inputs[\'Subsurface\'].default_value = 0.045\n    links.new(bsdf_leaf.outputs[\'BSDF\'], output.inputs[\'Surface\'])\n    try:\n        material.surface_render_method = \'DITHERED\'\n    except Exception:\n        material.blend_method = \'HASHED\'\n        material.shadow_method = \'HASHED\'\n    material.use_backface_culling = False\n    return material\n\nleaf_profiles = texture_manifest.get(\'leaf_profiles\', [])\nif not leaf_profiles:\n    leaf_profiles = [{\'id\':\'medium\', \'albedo\':\'leaf_albedo.png\'}]\nleaf_materials = [make_leaf_material(profile) for profile in leaf_profiles]\n\n# ---------- Trunk mesh ----------\nmesh = bpy.data.meshes.new(\'Tree_Trunk_Mesh\')\nmesh.from_pydata(vertices, [], faces)\nmesh.update(calc_edges=True)\ntrunk = bpy.data.objects.new(\'Tree_Trunk\', mesh)\nbpy.context.collection.objects.link(trunk)\ntrunk.data.materials.append(bark)\n\nuv_layer = mesh.uv_layers.new(name=\'UVMap\')\nfor poly in mesh.polygons:\n    poly.use_smooth = True\n    for loop_index in poly.loop_indices:\n        vi = mesh.loops[loop_index].vertex_index\n        uv_layer.data[loop_index].uv = (float(uvs[vi][0]), float(uvs[vi][1]))\n\n# ---------- Armature ----------\narm_data = bpy.data.armatures.new(\'Tree_Rig_Data\')\narm = bpy.data.objects.new(\'Tree_Rig\', arm_data)\nbpy.context.collection.objects.link(arm)\nbpy.context.view_layer.objects.active = arm\narm.select_set(True)\nbpy.ops.object.mode_set(mode=\'EDIT\')\nedit = {}\nfor b in rig[\'bones\']:\n    eb = arm_data.edit_bones.new(b[\'name\'])\n    eb.head = Vector(b[\'head\'])\n    eb.tail = Vector(b[\'tail\'])\n    if (eb.tail-eb.head).length < 1e-4:\n        eb.tail = eb.head + Vector((0,0,0.05))\n    edit[b[\'name\']] = eb\nfor b in rig[\'bones\']:\n    if b[\'parent_name\'] and b[\'parent_name\'] in edit:\n        edit[b[\'name\']].parent = edit[b[\'parent_name\']]\n        edit[b[\'name\']].use_connect = False\nbpy.ops.object.mode_set(mode=\'OBJECT\')\n\nbone_names = rig[\'bone_names\']\nfor name in bone_names:\n    trunk.vertex_groups.new(name=name)\nfor vi in range(len(vertices)):\n    for slot in range(4):\n        bi = int(skin_indices[vi][slot])\n        w = float(skin_weights[vi][slot])\n        if bi >= 0 and w > 0:\n            trunk.vertex_groups[bone_names[bi]].add([vi], w, \'REPLACE\')\nmod = trunk.modifiers.new(\'TreeArmature\', \'ARMATURE\')\nmod.object = arm\n\n# ---------- Leaf crossed-card mesh ----------\nrng = random.Random(1234)\nleaf_verts=[]; leaf_faces=[]; leaf_uv=[]; leaf_bone_indices=[]; leaf_face_materials=[]\nnode_to_bone = {}\nfor bidx, b in enumerate(rig[\'bones\']):\n    node_to_bone[int(b[\'node\'])] = bidx\n\n# nearest rig ancestor for leaves\ndef nearest_bone_node(node):\n    node = int(node)\n    while node not in node_to_bone and node >= 0:\n        node = int(parents[node])\n    return node if node in node_to_bone else 0\n\nfor li, center in enumerate(leaf_xyz):\n    normalized_height = max(0.0, min(1.0, float(center[2]) / max(1.0, max(v[2] for v in leaf_xyz))))\n    density_noise = rng.random()\n    if normalized_height > 0.78 and density_noise < 0.34:\n        profile_slot = min(len(leaf_materials)-1, 0)\n    elif normalized_height > 0.55 and density_noise < 0.56:\n        profile_slot = min(len(leaf_materials)-1, 1)\n    elif density_noise < 0.78:\n        profile_slot = min(len(leaf_materials)-1, 2)\n    elif density_noise < 0.92:\n        profile_slot = min(len(leaf_materials)-1, 3)\n    else:\n        profile_slot = min(len(leaf_materials)-1, 4)\n    physical_length = float(leaf_profiles[profile_slot].get(\'physical_length_m\', 0.12))\n    scale = max(0.10, min(0.48, physical_length * 2.35 * rng.uniform(0.88, 1.14)))\n    yaw = float(rng.uniform(0, 2*math.pi))\n    tilt = float(rng.uniform(-0.35, 0.35))\n    c, s = math.cos(yaw), math.sin(yaw)\n    right = Vector((c, s, 0.0))\n    up = Vector((-s*math.sin(tilt), c*math.sin(tilt), math.cos(tilt)))\n    forward = right.cross(up)\n    axes = [(right, up), (forward, up)]\n    base_index = len(leaf_verts)\n    for plane_index, (ax0, ax1) in enumerate(axes):\n        corners = [\n            center - ax0*scale*0.55 - ax1*scale,\n            center + ax0*scale*0.55 - ax1*scale,\n            center + ax0*scale*0.55 + ax1*scale,\n            center - ax0*scale*0.55 + ax1*scale,\n        ]\n        leaf_verts.extend(corners)\n        leaf_uv.extend([(0,0),(1,0),(1,1),(0,1)])\n        q = base_index + plane_index*4\n        leaf_faces.append((q,q+1,q+2,q+3))\n        leaf_face_materials.append(profile_slot)\n    leaf_bone_indices.extend([node_to_bone[nearest_bone_node(leaf_node_ids[li])]]*8)\n\nleaf_mesh = bpy.data.meshes.new(\'Tree_Leaves_Mesh\')\nleaf_mesh.from_pydata([tuple(v) for v in leaf_verts], [], leaf_faces)\nleaf_mesh.update(calc_edges=True)\nleaves = bpy.data.objects.new(\'Tree_Leaves\', leaf_mesh)\nbpy.context.collection.objects.link(leaves)\nfor material in leaf_materials:\n    leaves.data.materials.append(material)\nfor poly_index, polygon in enumerate(leaf_mesh.polygons):\n    polygon.material_index = int(leaf_face_materials[poly_index])\nluv = leaf_mesh.uv_layers.new(name=\'UVMap\')\nfor poly in leaf_mesh.polygons:\n    for loop_index in poly.loop_indices:\n        vi = leaf_mesh.loops[loop_index].vertex_index\n        luv.data[loop_index].uv = leaf_uv[vi]\n\nfor name in bone_names:\n    leaves.vertex_groups.new(name=name)\nfor vi, bi in enumerate(leaf_bone_indices):\n    leaves.vertex_groups[bone_names[int(bi)]].add([vi], 1.0, \'REPLACE\')\nlmod = leaves.modifiers.new(\'TreeArmature\', \'ARMATURE\')\nlmod.object = arm\n\n# ---------- Ground ----------\nbpy.ops.mesh.primitive_plane_add(size=35, location=(0,0,-0.26))\nground=bpy.context.object\nground.name=\'Ground\'\ngmat=bpy.data.materials.new(\'M_Ground\')\ngmat.use_nodes=True\ngbsdf=gmat.node_tree.nodes.get(\'Principled BSDF\')\ngbsdf.inputs[\'Base Color\'].default_value=(0.055,0.07,0.045,1)\ngbsdf.inputs[\'Roughness\'].default_value=0.92\nground.data.materials.append(gmat)\n\n# ---------- Camera and lights ----------\ndef look_at(obj, target):\n    direction = Vector(target) - obj.location\n    obj.rotation_euler = direction.to_track_quat(\'-Z\',\'Y\').to_euler()\n\nbpy.ops.object.camera_add(location=(11.7,-14.2,8.7))\ncamera=bpy.context.object\ncamera.data.lens=58\nlook_at(camera,(0,0,4.0))\nbpy.context.scene.camera=camera\n\nbpy.ops.object.light_add(type=\'AREA\', location=(5.5,-4.5,12.0))\nkey=bpy.context.object\nkey.data.energy=1250\nkey.data.shape=\'DISK\'\nkey.data.size=6.0\nlook_at(key,(0,0,4))\n\nbpy.ops.object.light_add(type=\'AREA\', location=(-7.0,-1.0,7.0))\nfill=bpy.context.object\nfill.data.energy=850\nfill.data.size=7.0\nlook_at(fill,(0,0,4.5))\n\nbpy.ops.object.light_add(type=\'AREA\', location=(1.5,6.5,10.5))\nrim=bpy.context.object\nrim.data.energy=1450\nrim.data.size=5.0\nlook_at(rim,(0,0,5.2))\n\nbpy.ops.object.light_add(type=\'SUN\', location=(0,0,12))\nsun=bpy.context.object\nsun.rotation_euler=(math.radians(28),math.radians(-22),math.radians(32))\nsun.data.energy=1.3\nsun.data.angle=math.radians(8)\n\nscene=bpy.context.scene\nscene.render.resolution_x=int(RENDER_CFG[\'resolution_x\'])\nscene.render.resolution_y=int(RENDER_CFG[\'resolution_y\'])\nscene.render.resolution_percentage=100\nscene.render.image_settings.file_format=\'PNG\'\nscene.render.film_transparent=bool(RENDER_CFG[\'transparent_background\'])\nscene.render.filepath=str(ROOT/\'tree_render.png\')\nscene.world.color=(0.018,0.025,0.035)\n\n# Engine compatibility across Blender versions.\nrequested=RENDER_CFG.get(\'engine\',\'EEVEE\').upper()\nif requested==\'CYCLES\':\n    scene.render.engine=\'CYCLES\'\n    scene.cycles.samples=int(RENDER_CFG.get(\'samples\',64))\n    try:\n        scene.cycles.device=\'GPU\'\n    except Exception:\n        pass\nelse:\n    try:\n        scene.render.engine=\'BLENDER_EEVEE_NEXT\'\n    except Exception:\n        scene.render.engine=\'BLENDER_EEVEE\'\n    try:\n        scene.eevee.taa_render_samples=int(RENDER_CFG.get(\'samples\',64))\n        scene.eevee.use_gtao=True\n        scene.eevee.gtao_distance=4\n        scene.eevee.gtao_factor=1.3\n    except Exception:\n        pass\n\n# Color management compatible with Blender 3.0.1 and newer. The apt Blender\n# build may not ship AgX or a Filmic OCIO role, so use Standard explicitly.\ntry:\n    scene.view_settings.view_transform = \'Standard\'\nexcept Exception:\n    pass\ntry:\n    available_looks = [item.name for item in bpy.types.ColorManagedViewSettings.bl_rna.properties[\'look\'].enum_items]\n    if \'Medium High Contrast\' in available_looks:\n        scene.view_settings.look = \'Medium High Contrast\'\n    elif \'None\' in available_looks:\n        scene.view_settings.look = \'None\'\nexcept Exception:\n    pass\n\n# Render still\nbpy.ops.wm.save_as_mainfile(filepath=str(ROOT/\'tree_asset.blend\'))\nbpy.ops.render.render(write_still=True)\nimport shutil\nshutil.copyfile(str(ROOT/\'tree_asset.blend\'), str(ROOT/\'asset_source.blend\'))\nshutil.copyfile(str(ROOT/\'tree_render.png\'), str(ROOT/\'asset_render.png\'))\n\n# Export only game-asset objects, not ground/camera/lights.\nbpy.ops.object.select_all(action=\'DESELECT\')\nfor obj in (trunk, leaves, arm):\n    obj.select_set(True)\nbpy.context.view_layer.objects.active = trunk\nbpy.ops.export_scene.gltf(\n    filepath=str(ROOT/\'tree_game_asset.glb\'),\n    export_format=\'GLB\',\n    use_selection=True,\n    export_animations=True,\n    export_yup=True,\n)\nshutil.copyfile(str(ROOT/\'tree_game_asset.glb\'), str(ROOT/\'game_asset.glb\'))\n\n# Optional camera turntable.\nif os.environ.get(\'TREE_TURNTABLE\',\'0\') == \'1\':\n    empty=bpy.data.objects.new(\'CameraOrbit\',None)\n    bpy.context.collection.objects.link(empty)\n    empty.location=(0,0,4)\n    camera.parent=empty\n    camera.matrix_parent_inverse=empty.matrix_world.inverted()\n    frames=int(RENDER_CFG.get(\'turntable_frames\',120))\n    empty.rotation_euler=(0,0,0)\n    empty.keyframe_insert(data_path=\'rotation_euler\',frame=1,index=2)\n    empty.rotation_euler=(0,0,2*math.pi)\n    empty.keyframe_insert(data_path=\'rotation_euler\',frame=frames,index=2)\n    if empty.animation_data and empty.animation_data.action:\n        for fc in empty.animation_data.action.fcurves:\n            for kp in fc.keyframe_points:\n                kp.interpolation=\'LINEAR\'\n    scene.frame_start=1; scene.frame_end=frames\n    scene.render.image_settings.file_format=\'FFMPEG\'\n    scene.render.ffmpeg.format=\'MPEG4\'\n    scene.render.ffmpeg.codec=\'H264\'\n    scene.render.ffmpeg.constant_rate_factor=\'MEDIUM\'\n    scene.render.filepath=str(ROOT/\'tree_turntable.mp4\')\n    bpy.ops.render.render(animation=True)\n'.replace('__ROOT__', str(ROOT))
generic_blender_script = 'import bpy, json, math, os, shutil\nfrom mathutils import Vector\nfrom pathlib import Path\n\nROOT = Path(r"__ROOT__")\nPLAN = json.loads((ROOT / "asset_graph.json").read_text(encoding="utf-8"))\nTEXTURES = json.loads((ROOT / "texture_manifest.json").read_text(encoding="utf-8"))\nRENDER_CFG = json.loads((ROOT / "request.json").read_text(encoding="utf-8"))["render"]\n\nbpy.ops.object.select_all(action="SELECT")\nbpy.ops.object.delete(use_global=False)\n\ndef safe_colorspace(image, requested="sRGB"):\n    data_request = requested not in ("sRGB", "Filmic sRGB")\n    candidates = ["Non-Color", "Raw", "Linear", "sRGB"] if data_request else ["sRGB", "Filmic sRGB", "Linear"]\n    for candidate in candidates:\n        try:\n            image.colorspace_settings.name = candidate\n            return candidate\n        except (TypeError, ValueError):\n            continue\n    return image.colorspace_settings.name\n\ndef image_node(nodes, path, name, colorspace="sRGB"):\n    tex = nodes.new("ShaderNodeTexImage")\n    tex.name = name\n    tex.image = bpy.data.images.load(str(path), check_existing=True)\n    safe_colorspace(tex.image, colorspace)\n    tex.extension = "REPEAT"\n    return tex\n\ntexture_by_material = {entry["id"]: entry for entry in TEXTURES["materials"]}\nmaterial_by_id = {}\n\nfor spec in PLAN["materials"]:\n    material = bpy.data.materials.new("M_" + spec["id"])\n    material.use_nodes = True\n    nodes = material.node_tree.nodes\n    links = material.node_tree.links\n    for node in list(nodes):\n        nodes.remove(node)\n    output = nodes.new("ShaderNodeOutputMaterial")\n    bsdf = nodes.new("ShaderNodeBsdfPrincipled")\n    bsdf.inputs["Base Color"].default_value = tuple(spec["base_color"]) + (1.0,)\n    bsdf.inputs["Roughness"].default_value = float(spec["roughness"])\n    bsdf.inputs["Metallic"].default_value = float(spec["metallic"])\n\n    record = texture_by_material.get(spec["id"])\n    if record and record.get("variants"):\n        pbr = record["variants"][0].get("pbr", {})\n        if pbr:\n            albedo = image_node(nodes, ROOT / pbr["albedo"], spec["id"] + "_albedo", "sRGB")\n            normal = image_node(nodes, ROOT / pbr["normal"], spec["id"] + "_normal", "Non-Color")\n            rough = image_node(nodes, ROOT / pbr["roughness"], spec["id"] + "_rough", "Non-Color")\n            normal_map = nodes.new("ShaderNodeNormalMap")\n            links.new(albedo.outputs["Color"], bsdf.inputs["Base Color"])\n            links.new(normal.outputs["Color"], normal_map.inputs["Color"])\n            links.new(normal_map.outputs["Normal"], bsdf.inputs["Normal"])\n            links.new(rough.outputs["Color"], bsdf.inputs["Roughness"])\n            if spec["transparent"] and "Alpha" in albedo.outputs:\n                links.new(albedo.outputs["Alpha"], bsdf.inputs["Alpha"])\n\n    links.new(bsdf.outputs["BSDF"], output.inputs["Surface"])\n    if spec["transparent"]:\n        try:\n            material.surface_render_method = "DITHERED"\n        except Exception:\n            material.blend_method = "HASHED"\n            material.shadow_method = "HASHED"\n        material.use_backface_culling = False\n    material_by_id[spec["id"]] = material\n\ndef add_primitive(kind):\n    if kind == "cube":\n        bpy.ops.mesh.primitive_cube_add(size=2)\n    elif kind in ("sphere", "uv_sphere"):\n        bpy.ops.mesh.primitive_uv_sphere_add(segments=32, ring_count=16)\n    elif kind == "ico_sphere":\n        bpy.ops.mesh.primitive_ico_sphere_add(subdivisions=3)\n    elif kind == "cylinder":\n        bpy.ops.mesh.primitive_cylinder_add(vertices=32, radius=1, depth=2)\n    elif kind == "cone":\n        bpy.ops.mesh.primitive_cone_add(vertices=32, radius1=1, radius2=0, depth=2)\n    elif kind == "plane":\n        bpy.ops.mesh.primitive_plane_add(size=2)\n    elif kind == "torus":\n        bpy.ops.mesh.primitive_torus_add(major_segments=48, minor_segments=16)\n    elif kind == "capsule":\n        bpy.ops.mesh.primitive_uv_sphere_add(segments=24, ring_count=12)\n    else:\n        raise RuntimeError("Unsupported trusted primitive: " + kind)\n    return bpy.context.object\n\ndef add_modifier(obj, modifier_spec):\n    kind = modifier_spec["type"]\n    strength = float(modifier_spec["strength"])\n    if kind == "bevel":\n        mod = obj.modifiers.new("Bevel", "BEVEL")\n        mod.width = max(0.001, min(0.5, strength))\n        mod.segments = 3\n    elif kind == "subdivision":\n        mod = obj.modifiers.new("Subdivision", "SUBSURF")\n        mod.levels = max(0, min(2, int(round(strength))))\n        mod.render_levels = mod.levels\n    elif kind == "solidify":\n        mod = obj.modifiers.new("Solidify", "SOLIDIFY")\n        mod.thickness = max(0.001, min(1.0, strength))\n    elif kind == "weighted_normal":\n        obj.modifiers.new("WeightedNormal", "WEIGHTED_NORMAL")\n    elif kind == "decimate":\n        mod = obj.modifiers.new("Decimate", "DECIMATE")\n        mod.ratio = max(0.05, min(1.0, strength))\n    # mirror and array are intentionally omitted in this first trusted compiler\n    # unless their extra parameters are explicitly added to the schema.\n\nobjects = {}\nfor part in PLAN["parts"]:\n    obj = add_primitive(part["primitive"])\n    obj.name = part["id"]\n    obj.location = tuple(float(v) for v in part["location"])\n    obj.rotation_euler = tuple(math.radians(float(v)) for v in part["rotation_deg"])\n    obj.scale = tuple(float(v) for v in part["scale"])\n    if part["material_id"] in material_by_id:\n        obj.data.materials.append(material_by_id[part["material_id"]])\n    for modifier in part.get("modifiers", []):\n        add_modifier(obj, modifier)\n    objects[part["id"]] = obj\n\nfor part in PLAN["parts"]:\n    parent_id = part.get("parent_id")\n    if parent_id:\n        objects[part["id"]].parent = objects[parent_id]\n\n# Ground is render-only.\nmax_dimension = max(float(v) for v in PLAN["dimensions_m"])\nbpy.ops.mesh.primitive_plane_add(size=max(12.0, max_dimension * 3.0), location=(0, 0, -0.02))\nground = bpy.context.object\nground.name = "Preview_Ground"\nground_mat = bpy.data.materials.new("M_Preview_Ground")\nground_mat.use_nodes = True\nground_bsdf = ground_mat.node_tree.nodes.get("Principled BSDF")\nground_bsdf.inputs["Base Color"].default_value = (0.045, 0.052, 0.055, 1)\nground_bsdf.inputs["Roughness"].default_value = 0.92\nground.data.materials.append(ground_mat)\n\ndef look_at(obj, target):\n    direction = Vector(target) - obj.location\n    obj.rotation_euler = direction.to_track_quat("-Z", "Y").to_euler()\n\ncenter_z = float(PLAN["dimensions_m"][2]) * 0.45\ncamera_distance = max(7.5, max_dimension * 2.8)\nbpy.ops.object.camera_add(location=(camera_distance, -camera_distance, camera_distance * 0.72))\ncamera = bpy.context.object\ncamera.data.lens = 58\nlook_at(camera, (0, 0, center_z))\nbpy.context.scene.camera = camera\n\nfor light_type, location, energy, size in [\n    ("AREA", (5.5, -4.5, 10.0), 1200, 6.0),\n    ("AREA", (-6.0, -1.0, 7.0), 750, 7.0),\n    ("AREA", (1.5, 6.5, 9.0), 1050, 5.0),\n]:\n    bpy.ops.object.light_add(type=light_type, location=location)\n    light = bpy.context.object\n    light.data.energy = energy\n    light.data.size = size\n    look_at(light, (0, 0, center_z))\n\nscene = bpy.context.scene\nscene.render.resolution_x = int(RENDER_CFG["resolution_x"])\nscene.render.resolution_y = int(RENDER_CFG["resolution_y"])\nscene.render.resolution_percentage = 100\nscene.render.image_settings.file_format = "PNG"\nscene.render.film_transparent = bool(RENDER_CFG["transparent_background"])\nscene.render.filepath = str(ROOT / "asset_render.png")\ntry:\n    scene.view_settings.view_transform = "Standard"\nexcept Exception:\n    pass\n\nengine = str(RENDER_CFG.get("engine", "EEVEE")).upper()\nscene.render.engine = "BLENDER_EEVEE" if engine == "EEVEE" else "BLENDER_EEVEE"\nscene.eevee.taa_render_samples = int(RENDER_CFG.get("samples", 64))\nscene.render.image_settings.color_mode = "RGBA" if RENDER_CFG["transparent_background"] else "RGB"\n\nbpy.ops.wm.save_as_mainfile(filepath=str(ROOT / "asset_source.blend"))\nbpy.ops.render.render(write_still=True)\n\nbpy.ops.object.select_all(action="DESELECT")\nfor obj in objects.values():\n    obj.select_set(True)\nif objects:\n    bpy.context.view_layer.objects.active = next(iter(objects.values()))\nbpy.ops.export_scene.gltf(\n    filepath=str(ROOT / "game_asset.glb"),\n    export_format="GLB",\n    use_selection=True,\n    export_animations=True,\n    export_yup=True,\n)\nprint("UNIVERSAL_ASSET_EXPORT_OK", PLAN["asset_id"])\n'.replace('__ROOT__', str(ROOT))
blender_script = tree_blender_script if IS_TREE else generic_blender_script
(ROOT / 'build_asset.py').write_text(blender_script, encoding='utf-8')
(ROOT / 'build_tree_asset.py').write_text(tree_blender_script, encoding='utf-8')
assert 'colorspace_settings.name = colorspace' not in blender_script
assert 'def safe_colorspace' in blender_script
assert 'import numpy' not in blender_script
assert 'np.' not in blender_script
compile(blender_script, '<trusted_blender_compiler>', 'exec')
print('Trusted compiler:', 'PolyFlow tree' if IS_TREE else 'Universal AssetGraph')
print('Wrote:', ROOT / 'build_asset.py')


In [ ]:
#@title Build, render, animate, and export the selected asset
common_inputs = [
    ROOT / "asset_graph.json",
    ROOT / "texture_manifest.json",
    ROOT / "build_asset.py",
    ROOT / "request.json",
]
tree_inputs = [
    ROOT / "tree_mesh.json.gz",
    ROOT / "skeleton.json.gz",
    ROOT / "rig_map.json",
    ROOT / "bark_albedo.png",
    ROOT / "bark_normal.png",
    ROOT / "bark_roughness.png",
]
required_inputs = common_inputs + (tree_inputs if IS_TREE else [])
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Required compiler inputs are missing. Run all preceding cells: "
        + json.dumps(missing_inputs, indent=2)
    )

builder_path = ROOT / "build_asset.py"
builder_text = builder_path.read_text(encoding="utf-8")
assert "import numpy" not in builder_text
assert "np." not in builder_text
assert "def safe_colorspace" in builder_text

env = os.environ.copy()
env["TREE_TURNTABLE"] = "1" if RENDER.create_turntable else "0"

preflight_expression = (
    "import bpy,gzip,json,random;"
    "from mathutils import Vector;"
    "print('POLYFLOW_BLENDER_PREFLIGHT_OK',bpy.app.version_string);"
    "img=bpy.data.images.new('colorspace_probe',width=1,height=1);"
    "print('BLENDER_COLORSPACES',img.colorspace_settings.bl_rna.properties['name'].enum_items.keys())"
)
preflight = subprocess.run(
    [
        "blender", "--background", "--factory-startup",
        "--python-expr", preflight_expression,
    ],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("\n".join(preflight.stdout.splitlines()[-40:]))
if preflight.returncode != 0 or "POLYFLOW_BLENDER_PREFLIGHT_OK" not in preflight.stdout:
    raise RuntimeError(
        f"Blender preflight failed with return code {preflight.returncode}"
    )

result = subprocess.run(
    [
        "blender", "--background", "--factory-startup",
        "--python", str(builder_path),
    ],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("\n".join(result.stdout.splitlines()[-180:]))

if result.returncode != 0:
    raise RuntimeError(
        f"Blender compiler failed with return code {result.returncode}. "
        "The final log lines are printed above."
    )

# Tree compiler writes legacy names and generic aliases. Generic compiler writes aliases directly.
if IS_TREE:
    alias_pairs = [
        (ROOT / "tree_render.png", ROOT / "asset_render.png"),
        (ROOT / "tree_game_asset.glb", ROOT / "game_asset.glb"),
        (ROOT / "tree_asset.blend", ROOT / "asset_source.blend"),
    ]
    for source, destination in alias_pairs:
        if source.exists() and not destination.exists():
            shutil.copyfile(source, destination)

expected_outputs = [
    ROOT / "asset_render.png",
    ROOT / "game_asset.glb",
    ROOT / "asset_source.blend",
]
missing_outputs = [str(path) for path in expected_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError(
        "Blender returned success but expected outputs are missing: "
        + json.dumps(missing_outputs, indent=2)
    )

for file_path in expected_outputs:
    print(file_path.name, f"{file_path.stat().st_size/1024/1024:.2f} MB")


## Asset preview

The next cell displays the rendered asset directly in Colab and also
creates downloadable preview files. It does not use Pillow.

Generated preview files:

- `tree_asset_screenshot.png` — full notebook preview
- `tree_asset_thumbnail.png` — compact repository thumbnail
- `tree_turntable_contact_sheet.png` — generated when turntable frames exist


In [ ]:
#@title Display asset screenshot in Colab and create preview files
from IPython.display import display, Image as IPyImage
import cv2
import numpy as np

render_path = ROOT / 'asset_render.png'
screenshot_path = ROOT / 'asset_screenshot.png'
thumbnail_path = ROOT / 'asset_thumbnail.png'
contact_sheet_path = ROOT / 'asset_turntable_contact_sheet.png'


def _read_preview(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise FileNotFoundError(f'Could not decode preview image: {path}')
    return image


def _fit_inside(image: np.ndarray, max_width: int, max_height: int) -> np.ndarray:
    height, width = image.shape[:2]
    scale = min(max_width / max(width, 1), max_height / max(height, 1), 1.0)
    if scale >= 1.0:
        return image.copy()
    resized = cv2.resize(
        image,
        (max(1, int(round(width * scale))), max(1, int(round(height * scale)))),
        interpolation=cv2.INTER_AREA,
    )
    return resized


def _checkerboard(height: int, width: int, block: int = 24) -> np.ndarray:
    yy, xx = np.indices((height, width))
    pattern = ((xx // block + yy // block) % 2).astype(np.uint8)
    light = np.full((height, width, 3), 212, dtype=np.uint8)
    dark = np.full((height, width, 3), 174, dtype=np.uint8)
    return np.where(pattern[..., None] == 0, light, dark)


def _composite_alpha(image: np.ndarray) -> np.ndarray:
    if image.ndim != 3 or image.shape[2] != 4:
        return image[..., :3] if image.ndim == 3 else cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    color = image[..., :3].astype(np.float32)
    alpha = image[..., 3:4].astype(np.float32) / 255.0
    background = _checkerboard(image.shape[0], image.shape[1]).astype(np.float32)
    return np.uint8(np.clip(color * alpha + background * (1.0 - alpha), 0, 255))


def _write_preview(path: Path, image: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(path), image):
        raise IOError(f'OpenCV failed to write preview: {path}')


if not render_path.exists():
    raise FileNotFoundError(
        f'{render_path.name} does not exist. Re-run the Blender build cell and '
        'confirm it finishes without an exception.'
    )

render = _read_preview(render_path)
render_bgr = _composite_alpha(render)

# Preserve a large high-quality screenshot for inspection.
screenshot = _fit_inside(render_bgr, max_width=1600, max_height=1200)
_write_preview(screenshot_path, screenshot)

# Create a compact repository/gallery thumbnail with a neutral border.
thumb_inner = _fit_inside(render_bgr, max_width=900, max_height=650)
border = 28
thumbnail = cv2.copyMakeBorder(
    thumb_inner,
    border,
    border,
    border,
    border,
    borderType=cv2.BORDER_CONSTANT,
    value=(28, 29, 34),
)
_write_preview(thumbnail_path, thumbnail)

# Search several likely turntable-frame layouts.
turntable_candidates = []
for folder in [
    ROOT / 'turntable',
    ROOT / 'turntable_frames',
    ROOT / 'frames',
    ROOT,
]:
    if folder.exists():
        for pattern in ('turntable_*.png', 'frame_*.png', 'tree_turn_*.png'):
            turntable_candidates.extend(sorted(folder.glob(pattern)))

# Deduplicate while retaining order.
seen = set()
turntable_frames = []
for candidate in turntable_candidates:
    resolved = str(candidate.resolve())
    if resolved in seen or candidate == render_path:
        continue
    seen.add(resolved)
    turntable_frames.append(candidate)

if turntable_frames:
    # Sample up to eight frames evenly across the animation.
    count = min(8, len(turntable_frames))
    indices = np.linspace(0, len(turntable_frames) - 1, count, dtype=int)
    sampled = [turntable_frames[int(i)] for i in indices]

    tiles = []
    for frame_path in sampled:
        frame = _composite_alpha(_read_preview(frame_path))
        tile = _fit_inside(frame, 360, 270)
        canvas = np.full((286, 376, 3), (28, 29, 34), dtype=np.uint8)
        y = (canvas.shape[0] - tile.shape[0]) // 2
        x = (canvas.shape[1] - tile.shape[1]) // 2
        canvas[y:y + tile.shape[0], x:x + tile.shape[1]] = tile
        cv2.putText(
            canvas,
            frame_path.stem[-18:],
            (12, canvas.shape[0] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.42,
            (224, 224, 224),
            1,
            cv2.LINE_AA,
        )
        tiles.append(canvas)

    columns = min(4, len(tiles))
    rows = int(np.ceil(len(tiles) / columns))
    empty = np.full_like(tiles[0], (28, 29, 34))
    while len(tiles) < rows * columns:
        tiles.append(empty.copy())

    row_images = [
        np.concatenate(tiles[row * columns:(row + 1) * columns], axis=1)
        for row in range(rows)
    ]
    contact_sheet = np.concatenate(row_images, axis=0)
    _write_preview(contact_sheet_path, contact_sheet)
else:
    print('No turntable frame sequence found; the main screenshot is still available.')

print('Asset screenshot:', screenshot_path)
print('Asset thumbnail:', thumbnail_path)
if contact_sheet_path.exists():
    print('Turntable contact sheet:', contact_sheet_path)

print('\nRendered asset preview:')
display(IPyImage(filename=str(screenshot_path), width=900))

print('Repository thumbnail:')
display(IPyImage(filename=str(thumbnail_path), width=600))

if contact_sheet_path.exists():
    print('Turntable contact sheet:')
    display(IPyImage(filename=str(contact_sheet_path), width=1000))


In [ ]:
#@title Download the preview images and generated asset
from google.colab import files

download_targets = [
    ROOT / 'asset_screenshot.png',
    ROOT / 'asset_thumbnail.png',
    ROOT / 'asset_turntable_contact_sheet.png',
    ROOT / 'game_asset.glb',
    ROOT / 'asset_source.blend',
]

available_downloads = [path for path in download_targets if path.exists()]
if not available_downloads:
    raise FileNotFoundError('No preview or asset files are available to download.')

for path in available_downloads:
    print('Preparing download:', path.name)
    files.download(str(path))


In [ ]:
#@title Fast repair self-tests (no GPU, no API key, no Blender required)
import ast
import tempfile


def _assert_png(path: Path, channels: int | None = None):
    data = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    assert data is not None, f'PNG decode failed: {path}'
    assert data.shape[0] > 0 and data.shape[1] > 0
    if channels is not None:
        actual = 1 if data.ndim == 2 else data.shape[2]
        assert actual == channels, (path, actual, channels)
    return data


def run_repair_self_tests():
    # Runtime backend check. Static notebook validation also verifies that no
    # production cell imports or calls Pillow.
    assert cv2 is not None
    assert np is not None
    assert 'PIL' not in procedural_bark.__code__.co_names
    assert 'PIL' not in procedural_leaf.__code__.co_names
    assert 'PIL' not in derive_pbr.__code__.co_names

    with tempfile.TemporaryDirectory() as temp_dir:
        temp = Path(temp_dir)
        bark_a = temp / 'bark_a.png'
        bark_b = temp / 'bark_b.png'
        leaf = temp / 'leaf.png'

        procedural_bark(bark_a, size=256, seed=123)
        procedural_bark(bark_b, size=256, seed=123)
        procedural_leaf(leaf, size=256, seed=456)

        bark_arr = _assert_png(bark_a, 3)
        _assert_png(leaf, 4)
        assert bark_a.read_bytes() == bark_b.read_bytes(), 'Bark fallback is not deterministic'

        tiled = mirror_tileable_array(bark_arr)
        assert tiled.shape == bark_arr.shape
        assert np.mean(np.abs(tiled[:, 0].astype(float) - tiled[:, -1].astype(float))) < 12.0
        assert np.mean(np.abs(tiled[0].astype(float) - tiled[-1].astype(float))) < 12.0

        previous_root = globals()['ROOT']
        try:
            globals()['ROOT'] = temp
            normal_path, roughness_path = derive_pbr(bark_a)
            normal = _assert_png(normal_path, 3)
            roughness = _assert_png(roughness_path, 1)
            assert normal.shape[:2] == bark_arr.shape[:2]
            assert roughness.shape[:2] == bark_arr.shape[:2]
        finally:
            globals()['ROOT'] = previous_root

    # Regression checks for Blender compatibility and secure routing.
    blender_source = blender_script
    assert 'def safe_colorspace' in blender_source
    assert 'colorspace_settings.name = colorspace' not in blender_source
    assert 'import numpy' not in blender_source
    assert 'np.' not in blender_source
    compile(blender_source, '<blender_script>', 'exec')
    validate_asset_graph(json.loads(json.dumps(ASSET_PLAN)))
    assert SECURITY.execute_model_python is False
    if IS_TREE:
        assert len(ASSET_PLAN['texture_policy']['leaf_profiles']) >= 4
        assert len(TEXTURE_MANIFEST['leaf_profiles']) >= 4

    print('PASS: Pillow removed')
    print('PASS: deterministic procedural bark')
    print('PASS: RGBA procedural leaf')
    print('PASS: tileable and PBR map generation')
    print('PASS: Blender color-space compatibility routing')
    print('PASS: Blender builder has no NumPy dependency')
    print('PASS: secure AssetGraph validation')
    print('PASS: multi-scale leaf texture profiles')
    print('PASS: compressed JSON Blender bridge')
    print('PASS: embedded Blender script compiles')
    assert callable(_fit_inside)
    assert callable(_composite_alpha)
    assert callable(_write_preview)
    print('PASS: notebook screenshot and thumbnail helpers')


run_repair_self_tests()


## 6. RigFlow-Tree: a more advanced learned rigging-surface model

The procedural mapping above is immediately usable. The following research architecture goes further by jointly learning topology, bone affinity, deformation behavior, and material stiffness.

### Proposed state

For vertex `i` and bone `b`:

\[
\mathbf{q}_i=
[\mathbf{p}_i,\mathbf{n}_i,\mathbf{e}^{topo}_i,
\mathbf{e}^{geo}_i,\kappa_i,\rho_i,s_i],
\qquad
\mathbf{h}_b=[\mathbf{a}_b,\mathbf{t}_b,\ell_b,r_b,o_b].
\]

A surface Transformer processes all vertex states while a skeleton Transformer processes bones. Cross-attention predicts sparse skin weights, local frames, stiffness, and a flow velocity over both surface and rig channels.

### Multi-objective training

\[
\mathcal{L}_{total}=
\lambda_{flow}\mathcal{L}_{flow}
+\lambda_{bind}\mathcal{L}_{bind}
+\lambda_{pose}\mathcal{L}_{pose}
+\lambda_{arap}\mathcal{L}_{ARAP}
+\lambda_{topo}\mathcal{L}_{edge}
+\lambda_{simplex}\mathcal{L}_{simplex}
+\lambda_{sparse}\mathcal{L}_{sparse}
+\lambda_{lineage}\mathcal{L}_{lineage}.
\]

- `flow`: rectified-flow velocity over geometry, topology, and rig channels.
- `bind`: reconstruct the rest surface under predicted bind transforms.
- `pose`: reconstruct deformed training poses.
- `ARAP`: preserve local edge lengths and branch volume.
- `edge`: recover surface adjacency from continuous topology coordinates.
- `simplex`: ensure nonnegative weights sum to one.
- `sparse`: encourage at most four meaningful influences.
- `lineage`: prevent weights from bleeding across unrelated branches that happen to be spatially close.

For trees, this can be trained on synthetic wind, gravity, pruning, and impact poses generated automatically in Blender—removing the need for hand-rigged training data.


In [ ]:
if IS_TREE:
    #@title RigFlow-Tree neural blueprint (forward-pass runnable)
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class FourierTime(nn.Module):
        def __init__(self, dim=128):
            super().__init__()
            self.freq = nn.Parameter(torch.randn(dim//2), requires_grad=False)
        def forward(self, t):
            x = t[..., None] * self.freq[None, :] * 2 * math.pi
            return torch.cat([x.sin(), x.cos()], dim=-1)

    class RigFlowTree(nn.Module):
        """Research blueprint for joint surface/topology/rig flow prediction."""
        def __init__(self, vertex_dim, bone_dim=11, hidden=256, heads=8, layers=6, topo_dim=32):
            super().__init__()
            self.time = FourierTime(hidden)
            self.vertex_in = nn.Linear(vertex_dim, hidden)
            self.bone_in = nn.Linear(bone_dim, hidden)
            enc_layer = nn.TransformerEncoderLayer(hidden, heads, hidden*4, batch_first=True, norm_first=True)
            self.surface_encoder = nn.TransformerEncoder(enc_layer, layers)
            bone_layer = nn.TransformerEncoderLayer(hidden, heads, hidden*4, batch_first=True, norm_first=True)
            self.bone_encoder = nn.TransformerEncoder(bone_layer, max(2, layers//2))
            self.cross = nn.MultiheadAttention(hidden, heads, batch_first=True)
            self.velocity = nn.Linear(hidden, 3 + 3 + topo_dim + 1 + 6)  # xyz, normal, topology, stiffness, 6D frame
            self.weight_q = nn.Linear(hidden, hidden, bias=False)
            self.weight_k = nn.Linear(hidden, hidden, bias=False)
            self.anchor = nn.Linear(hidden, 3)

        def forward(self, vertex_state, bone_state, t, topk=4):
            # vertex_state: [B,V,C], bone_state: [B,J,Cb], t: [B]
            tv = self.time(t)[:, None, :]
            v = self.vertex_in(vertex_state) + tv
            b = self.bone_in(bone_state) + tv
            v = self.surface_encoder(v)
            b = self.bone_encoder(b)
            attended, _ = self.cross(v, b, b, need_weights=False)
            h = v + attended
            velocity = self.velocity(h)
            anchor_offset = self.anchor(h)

            logits = torch.einsum('bvh,bjh->bvj', self.weight_q(h), self.weight_k(b)) / math.sqrt(h.shape[-1])
            if topk < logits.shape[-1]:
                values, indices = torch.topk(logits, k=topk, dim=-1)
                sparse_logits = torch.full_like(logits, -1e4)
                sparse_logits.scatter_(-1, indices, values)
                weights = sparse_logits.softmax(dim=-1)
            else:
                weights = logits.softmax(dim=-1)
            return {'velocity': velocity, 'weights': weights, 'anchor_offset': anchor_offset}

    # Build compact bone descriptors from the generated rig.
    bone_state_np=[]
    for b in bones:
        head=np.asarray(b['head'],np.float32); tail=np.asarray(b['tail'],np.float32)
        tangent=normalize(tail-head).astype(np.float32)
        length=np.linalg.norm(tail-head)
        node=int(b['node'])
        bone_state_np.append(np.concatenate([head,tangent,[length,radii[node],orders[node],node_geodesic[node],node_stiffness[node]]]))
    bone_state_np=np.asarray(bone_state_np,np.float32)

    # Validate a bounded forward pass rather than sending the full mesh through a quadratic Transformer.
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model=RigFlowTree(vertex_dim=surface_state.shape[1], bone_dim=bone_state_np.shape[1]).to(device)
    V=min(768,len(surface_state))
    with torch.no_grad():
        out=model(
            torch.from_numpy(surface_state[:V])[None].to(device),
            torch.from_numpy(bone_state_np)[None].to(device),
            torch.tensor([0.5],device=device),
            topk=4,
        )
    print({k:tuple(v.shape) for k,v in out.items()})
    print('Max influences per vertex:', int((out['weights'][0] > 1e-6).sum(-1).max()))

else:
    print('Skipped tree-only stage: #@title RigFlow-Tree neural blueprint')


In [ ]:
if IS_TREE:
    #@title Example differentiable rig losses
    def rigflow_losses(pred, target_velocity, posed_pred=None, posed_target=None,
                       rest_edges=None, rest_vertices=None, deformed_vertices=None,
                       lineage_mask=None):
        velocity_loss = F.mse_loss(pred['velocity'], target_velocity)
        w = pred['weights']
        simplex = (w.sum(dim=-1) - 1.0).square().mean() + F.relu(-w).mean()
        entropy = -(w.clamp_min(1e-8) * w.clamp_min(1e-8).log()).sum(dim=-1).mean()

        pose_loss = torch.tensor(0.0, device=w.device)
        if posed_pred is not None and posed_target is not None:
            pose_loss = F.smooth_l1_loss(posed_pred, posed_target)

        arap = torch.tensor(0.0, device=w.device)
        if rest_edges is not None and rest_vertices is not None and deformed_vertices is not None:
            i, j = rest_edges[...,0], rest_edges[...,1]
            rest_len = (rest_vertices[:,i]-rest_vertices[:,j]).norm(dim=-1)
            def_len = (deformed_vertices[:,i]-deformed_vertices[:,j]).norm(dim=-1)
            arap = F.smooth_l1_loss(def_len, rest_len)

        lineage = torch.tensor(0.0, device=w.device)
        if lineage_mask is not None:
            # lineage_mask=1 where a vertex-bone assignment is biologically/topologically valid.
            lineage = (w * (1.0-lineage_mask)).mean()

        total = velocity_loss + 0.1*simplex + 0.01*entropy + 2.0*pose_loss + 0.5*arap + 0.5*lineage
        return {
            'total': total,
            'velocity': velocity_loss,
            'simplex': simplex,
            'weight_entropy': entropy,
            'pose': pose_loss,
            'arap': arap,
            'lineage': lineage,
        }

    print('Loss blueprint ready.')

else:
    print('Skipped tree-only stage: #@title Example differentiable rig losses')


## 7. Asset validation and packaging

The validation checks basic mesh sanity, finite feature values, skin-weight normalization, and required exports. The final ZIP contains the notebook outputs, source Blender script, textures, rig metadata, surface-comprehension tensors, GLB, and render.


## Universal-generation architecture

The notebook intentionally separates planning from execution:

1. **Prompt normalization** — constrain scale, category, style and engine target.
2. **GPT‑5.6 AssetGraph planning** — strict JSON schema.
3. **Security validation** — bounded counts, finite numbers, approved primitives,
   approved modifiers, safe identifiers and no remote URLs.
4. **Texture decomposition** — one coherent prompt family per material.
5. **Multi-scale foliage synthesis** — physical leaf size controls card scale.
6. **Trusted geometry compiler** — repository-owned Blender Python only.
7. **Topology and rig analysis** — PolyFlow/RigFlow path for trees.
8. **PBR derivation** — albedo, normal, roughness, AO and metallic without Pillow.
9. **Godot export** — GLB, animations, Y-up, meter scale and manifests.
10. **Validation and packaging** — render, source, checksums and ZIP.

GPT‑5.6 is used as a **planner**, not an unrestricted code executor. This makes
the workflow reusable for arbitrary prompts while keeping the executable
surface deterministic and reviewable.


In [ ]:
#@title Validate and package the universal deliverables
validation = {
    "asset_id": ASSET_PLAN["asset_id"],
    "category": ASSET_PLAN["category"],
    "strategy": ASSET_PLAN["strategy"],
    "planner_used": PLANNER_USED,
    "material_count": len(ASSET_PLAN["materials"]),
    "part_count": len(ASSET_PLAN["parts"]),
    "texture_count": sum(
        len(material["variants"]) for material in TEXTURE_MANIFEST["materials"]
    ) + len(TEXTURE_MANIFEST["leaf_profiles"]),
    "glb_exists": (ROOT / "game_asset.glb").exists(),
    "render_exists": (ROOT / "asset_render.png").exists(),
    "blend_exists": (ROOT / "asset_source.blend").exists(),
    "no_model_python_execution": not SECURITY.execute_model_python,
    "pillow_removed": True,
}
if IS_TREE:
    validation.update({
        "finite_vertices": bool(np.isfinite(vertices).all()),
        "finite_surface_state": bool(np.isfinite(surface_state).all()),
        "face_index_min": int(faces.min()),
        "face_index_max": int(faces.max()),
        "vertex_count": int(len(vertices)),
        "triangle_count": int(len(faces)),
        "bone_count": int(len(bones)),
        "skin_weight_sum_max_error": float(
            np.abs(skin_weights_array.sum(axis=1) - 1.0).max()
        ),
    })

for required in ("glb_exists", "render_exists", "blend_exists"):
    if not validation[required]:
        raise RuntimeError(f"Validation failed: {required}")

(ROOT / "validation.json").write_text(
    json.dumps(validation, indent=2),
    encoding="utf-8",
)
print(json.dumps(validation, indent=2))

zip_name = f"Universal_Asset_{ASSET_PLAN['asset_id']}.zip"
zip_path = Path("/content") / zip_name
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in ROOT.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=f"{ASSET_PLAN['asset_id']}/{path.relative_to(ROOT)}",
            )

print("Created:", zip_path, f"({zip_path.stat().st_size/1024/1024:.2f} MB)")


In [ ]:
#@title Download packaged universal asset
from google.colab import files
files.download(str(zip_path))
